In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────
# FILE PATHS — update for your machine
# ──────────────────────────────────────────────────────────────
FILE_SAT = r"C:\Users\Ibrahim\Desktop\Github_test\Forecasted_Impact.xlsx"
FILE_DDM = r"C:\Users\Ibrahim\Desktop\Github_test\Remal_ddm.xlsx"
OUT_DIR  = r"C:\Users\Ibrahim\Desktop\Github_test\results\AUC"
CYCLONE  = "Remal"

F1_SHEET       = "Remal"
F2_HOUSE_SHEET = "House"
F2_AGRI_SHEET  = "Agriculture"

os.makedirs(OUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────
# ROC / AUC — implemented from first principles (NumPy only)
# ──────────────────────────────────────────────────────────────
def roc_curve(scores, labels):
    """
    scores : array of continuous forecast values (e.g. satellite impact score)
    labels : array of 0/1 ground-truth (e.g. DDM damage > 0)

    Returns fpr, tpr by sweeping every unique score value as a
    candidate threshold.
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)

    P = labels.sum()
    N = len(labels) - P
    if P == 0 or N == 0:
        return None, None  # can't define a ROC curve without both classes

    thresholds = np.sort(np.unique(scores))[::-1]  # high -> low
    thresholds = np.concatenate(([np.inf], thresholds, [-np.inf]))

    tpr_list, fpr_list = [], []
    for t in thresholds:
        pred_pos = scores >= t
        tp = np.sum(pred_pos & (labels == 1))
        fp = np.sum(pred_pos & (labels == 0))
        tpr_list.append(tp / P)
        fpr_list.append(fp / N)

    return np.array(fpr_list), np.array(tpr_list)


def auc_score(fpr, tpr):
    """Trapezoidal-rule area under the ROC curve.
    Implemented manually (rather than via np.trapz/np.trapezoid) so
    this works identically across NumPy versions, since np.trapz was
    removed in NumPy 2.0 in favour of np.trapezoid."""
    if fpr is None:
        return np.nan
    order = np.argsort(fpr)
    x = fpr[order]
    y = tpr[order]
    return float(np.sum((x[1:] - x[:-1]) * (y[1:] + y[:-1]) / 2.0))


# ──────────────────────────────────────────────────────────────
# LOAD & CLEAN
# ──────────────────────────────────────────────────────────────
def clean_admin(x):
    if pd.isna(x): return np.nan
    return " ".join(str(x).strip().split())

f1 = pd.read_excel(FILE_SAT, sheet_name=F1_SHEET)
f1.columns = [str(c).strip() for c in f1.columns]
f1 = (f1.dropna(subset=["District", "Upazila"])
        .drop_duplicates(subset=["District", "Upazila"], keep="first")
        .reset_index(drop=True))

# House sheet: District, Upazila, col F (No_Total), col J (Amt_Total)
raw_house = pd.read_excel(FILE_DDM, sheet_name=F2_HOUSE_SHEET, header=None)
f2h = raw_house.iloc[2:, [0, 1, 5, 9]].copy().reset_index(drop=True)
f2h.columns = ["District", "Upazila", "No_Total", "Amt_Total"]
f2h["District"] = f2h["District"].ffill()
f2h = f2h.dropna(subset=["District", "Upazila"])
for c in ["No_Total", "Amt_Total"]:
    f2h[c] = pd.to_numeric(f2h[c], errors="coerce").fillna(0)

# Agriculture sheet: District, Upazila, col G (Total_Loss_Land), col H (Total_Loss_Amt)
raw_agri = pd.read_excel(FILE_DDM, sheet_name=F2_AGRI_SHEET, header=None)
f2a = raw_agri.iloc[2:, [0, 1, 6, 7]].copy().reset_index(drop=True)
f2a.columns = ["District", "Upazila", "Total_Loss_Land", "Total_Loss_Amt"]
f2a["District"] = f2a["District"].ffill()
f2a = f2a.dropna(subset=["District", "Upazila"])
for c in ["Total_Loss_Land", "Total_Loss_Amt"]:
    f2a[c] = pd.to_numeric(f2a[c], errors="coerce").fillna(0)

for df in [f1, f2h, f2a]:
    df["District"] = df["District"].apply(clean_admin)
    df["Upazila"] = df["Upazila"].apply(clean_admin)

m_house = pd.merge(f1, f2h, on=["District", "Upazila"], how="inner")
m_agri = pd.merge(f1, f2a, on=["District", "Upazila"], how="inner")
print(f"Merged -> House: {len(m_house)}   Agri: {len(m_agri)}\n")

# ──────────────────────────────────────────────────────────────
# ALL FOUR FIELDS
# ──────────────────────────────────────────────────────────────
VARS = [
    dict(df=m_house, impact_col="Norm_Impact_House", damage_col="No_Total",
         label="Total Number of Damaged Household"),
    dict(df=m_house, impact_col="Norm_Impact_House", damage_col="Amt_Total",
         label="Monitory Damage for Households"),
    dict(df=m_agri, impact_col="Norm_Impact_fAPAR", damage_col="Total_Loss_Land",
         label="Total Agriculture Land Area Damaged"),
    dict(df=m_agri, impact_col="Norm_Impact_fAPAR", damage_col="Total_Loss_Amt",
         label="Monitory Damage for Agricultural Land"),
]

# ──────────────────────────────────────────────────────────────
# COMPUTE + PLOT  (2x2 grid, one panel per field)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 12))
axes = axes.flatten()

print(f"Cyclone {CYCLONE} — ROC / AUC summary")
print("=" * 55)

for ax, v in zip(axes, VARS):
    scores = pd.to_numeric(v["df"][v["impact_col"]], errors="coerce")
    labels = (pd.to_numeric(v["df"][v["damage_col"]], errors="coerce").fillna(0) > 0).astype(int)
    valid = scores.notna()
    scores, labels = scores[valid].values, labels[valid].values

    fpr, tpr = roc_curve(scores, labels)
    auc = auc_score(fpr, tpr)
    print(f"{v['label']:42s}  n={len(scores):4d}  AUC={auc:.3f}")

    if fpr is None:
        ax.text(0.5, 0.5, "Insufficient class variation\n(all damaged or all undamaged)",
                ha="center", va="center", fontsize=10)
        ax.set_title(v["label"], fontsize=11, fontweight="bold")
        continue

    ax.plot(fpr, tpr, color="#1565C0", lw=2.2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], color="grey", lw=1.2, ls="--", label="No skill (AUC=0.5)")
    ax.fill_between(fpr, tpr, alpha=0.08, color="#1565C0")
    ax.set_xlabel("False Positive Rate  (False Alarms / Actual Negatives)")
    ax.set_ylabel("True Positive Rate  (POD)")
    ax.set_title(v["label"], fontsize=11, fontweight="bold")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.25)

fig.suptitle(f"",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
out_path = os.path.join(OUT_DIR, f"{CYCLONE}_ROC_AUC_AllFields.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"\nSaved: {out_path}")

Merged -> House: 125   Agri: 125

Cyclone Remal — ROC / AUC summary
Total Number of Damaged Household           n= 125  AUC=0.878
Monitory Damage for Households              n= 125  AUC=0.878
Total Agriculture Land Area Damaged         n= 125  AUC=0.623
Monitory Damage for Agricultural Land       n= 125  AUC=0.623

Saved: C:\Users\Ibrahim\Desktop\Github_test\results\AUC\Remal_ROC_AUC_AllFields.png


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.cell import column_index_from_string
import os
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 1. FILE PATHS
# ============================================================

FILE1 = r"C:\Users\Ibrahim\Desktop\Github_test\Forecasted_Impact.xlsx"
FILE2 = r"C:\Users\Ibrahim\Desktop\Github_test\Remal_ddm.xlsx"

OUT_DIR = r"C:\Users\Ibrahim\Desktop\Github_test\results"

OUT_EXCEL = os.path.join(
    OUT_DIR,
    "Remal_Validation_With_Balanced_Threshold_Optimization.xlsx"
)

F1_SHEET = "Remal"
F2_HOUSE_SHEET = "House"
F2_AGRI_SHEET = "Agriculture"

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 2. DISPLAY LABEL OPTIONS
# ============================================================

HOUSE_X_DISPLAY_LABEL = "Impacted Forecast for House Damaged"
AGRI_X_DISPLAY_LABEL = "Impacted Forecast for Agriculture"

#HOUSE_DAMAGE_COUNT_TITLE = "Normalized Impact of House vs Number of Damaged Houses"
#HOUSE_REPAIR_AMOUNT_TITLE = "Normalized Impact of House vs House Repair Amount"

#AGRI_LAND_DAMAGE_TITLE = "Normalized Impact of fAPAR vs Agricultural Land Damage"
#AGRI_DAMAGE_AMOUNT_TITLE = "Normalized Impact of fAPAR vs Agricultural Damage Amount"

Y_LABEL_NO_BRICK = "Number of Damaged Brick-built House"
Y_LABEL_NO_HALFBRICK = "Number of Damaged Half Brick-built House"
Y_LABEL_NO_RAW = "Number of Damaged Kacha House"
Y_LABEL_NO_TOTAL = "Total number of House Damaged"

Y_LABEL_AMT_BRICK = "Monitory Damage for Brick-built House"
Y_LABEL_AMT_HALFBRICK = "Monitory Damage for Half Brick-built House"
Y_LABEL_AMT_RAW = "Monitory Damage for Kacha House"
Y_LABEL_AMT_TOTAL = "Total Monitory Damage for House"

Y_LABEL_FULLY_LAND = "Agricultural Land Area Damaged (Fully Affected)"
Y_LABEL_PARTIAL_LAND = "Agricultural Land Area Damaged (Partially Affected)"
Y_LABEL_TOTAL_LOSS_LAND = "Agricultural Land Area Damaged (Total Affected)"

Y_LABEL_FULLY_AMT = "Monitory Damaged for Agricultural Land (Fully Affected)"
Y_LABEL_PARTIAL_AMT = "Monitory Damaged for Agricultural Land (Partially Affected)"
Y_LABEL_TOTAL_LOSS_AMT = "Monitory Damaged for Agricultural Land (Total Affected)"

LABEL_BRICK = "Brick-built House"
LABEL_HALFBRICK = "Half Brick-built House"
LABEL_RAW_HOUSE = "Kacha House"
LABEL_TOTAL_HOUSE = "Total Damaged Houses"

LABEL_AMT_BRICK = "Amount Spent on Brick-built House"
LABEL_AMT_HALFBRICK = "Amount Spent on Half Brick-built House"
LABEL_AMT_RAW = "Amount Spent on Kacha/Raw House"
LABEL_AMT_TOTAL = "Total Amount Spent on Repairs"

LABEL_FULLY_LAND = "Fully Damaged Land"
LABEL_PARTIAL_LAND = "Partially Damaged Land"
LABEL_TOTAL_LAND = "Total Damaged Land"

LABEL_FULLY_AMT = "Fully Damaged Amount"
LABEL_PARTIAL_AMT = "Partially Damaged Amount"
LABEL_TOTAL_AMT = "Total Damage Amount"

CLASS_X_LABEL = "Forecasted Impact Catagory"

# ============================================================
# 2B. THRESHOLD OPTIMIZATION SETTINGS
# ============================================================

THRESHOLD_MIN = 0.00
THRESHOLD_MAX = 1.00
THRESHOLD_STEP = 0.01

BIAS_MIN_ACCEPTABLE = 0.80
BIAS_MAX_ACCEPTABLE = 1.20
FAR_MAX_ACCEPTABLE = 0.25

DECISION_SCORE_NAME = "Decision Score"
DECISION_RULE_DESCRIPTION = (
    "Best threshold selected by max(CSI - FAR - abs(Bias - 1)), "
    "preferring Bias 0.80-1.20 and FAR <= 0.25"
)

# ============================================================
# 3. HELPER: EXCEL COLUMN LETTER TO PANDAS COLUMN NAME
# ============================================================

def excel_col_to_pandas_col(df, excel_col_letter):
    idx = column_index_from_string(excel_col_letter) - 1

    if idx < 0 or idx >= len(df.columns):
        raise ValueError(
            f"Excel column {excel_col_letter} is outside dataframe column range."
        )

    return df.columns[idx]

# ============================================================
# 4. LOAD SATELLITE DATA
# ============================================================

print("Loading satellite data...")

f1 = pd.read_excel(FILE1, sheet_name=F1_SHEET)
f1.columns = [str(c).strip() for c in f1.columns]

f1 = f1.dropna(how="all").copy()
f1 = f1.dropna(subset=["District", "Upazila"]).copy()

f1 = f1.drop_duplicates(
    subset=["District", "Upazila"],
    keep="first"
).reset_index(drop=True)

AGRI_CLASS_COL = excel_col_to_pandas_col(f1, "X")
HOUSE_CLASS_COL = excel_col_to_pandas_col(f1, "AG")

print(f"Satellite rows: {len(f1)}")
print("Agriculture/fAPAR class column from Excel X:", AGRI_CLASS_COL)
print("House class column from Excel AG:", HOUSE_CLASS_COL)

print("\nUnique fAPAR/agriculture classes:")
print(f1[AGRI_CLASS_COL].dropna().astype(str).str.strip().unique())

print("\nUnique house classes:")
print(f1[HOUSE_CLASS_COL].dropna().astype(str).str.strip().unique())

# ============================================================
# 5. LOAD DDM HOUSE DAMAGE DATA
# ============================================================

print("\nLoading DDM house damage data...")

raw_house = pd.read_excel(FILE2, sheet_name=F2_HOUSE_SHEET, header=None)

f2h = raw_house.iloc[2:].copy().reset_index(drop=True)
f2h = f2h.iloc[:, :10].copy()

# DDM House sheet mapping:
# C = No_Brick
# D = No_HalfBrick
# E = No_Raw
# F = No_Total
# G = Amt_Brick
# H = Amt_HalfBrick
# I = Amt_Raw
# J = Amt_Total

f2h.columns = [
    "District", "Upazila",
    "No_Brick", "No_HalfBrick", "No_Raw", "No_Total",
    "Amt_Brick", "Amt_HalfBrick", "Amt_Raw", "Amt_Total"
]

f2h["District"] = f2h["District"].ffill()
f2h = f2h.dropna(subset=["District", "Upazila"]).copy()

for c in f2h.columns[2:]:
    f2h[c] = pd.to_numeric(f2h[c], errors="coerce").fillna(0)

print(f"DDM house rows: {len(f2h)}")

# ============================================================
# 6. LOAD DDM AGRICULTURE DAMAGE DATA
# ============================================================

print("\nLoading DDM agriculture damage data...")

raw_agri = pd.read_excel(FILE2, sheet_name=F2_AGRI_SHEET, header=None)

f2a = raw_agri.iloc[2:].copy().reset_index(drop=True)

# DDM Agriculture sheet mapping:
# C = Fully_Land
# D = Fully_Amt
# E = Partial_Land
# F = Partial_Amt
# G = Total_Loss_Land
# H = Total_Loss_Amt

if f2a.shape[1] >= 8:
    f2a = f2a.iloc[:, :8].copy()
    f2a.columns = [
        "District", "Upazila",
        "Fully_Land", "Fully_Amt",
        "Partial_Land", "Partial_Amt",
        "Total_Loss_Land", "Total_Loss_Amt"
    ]
else:
    f2a = f2a.iloc[:, :6].copy()
    f2a.columns = [
        "District", "Upazila",
        "Fully_Land", "Fully_Amt",
        "Partial_Land", "Partial_Amt"
    ]

    f2a["Total_Loss_Land"] = (
        pd.to_numeric(f2a["Fully_Land"], errors="coerce").fillna(0) +
        pd.to_numeric(f2a["Partial_Land"], errors="coerce").fillna(0)
    )

    f2a["Total_Loss_Amt"] = (
        pd.to_numeric(f2a["Fully_Amt"], errors="coerce").fillna(0) +
        pd.to_numeric(f2a["Partial_Amt"], errors="coerce").fillna(0)
    )

f2a["District"] = f2a["District"].ffill()
f2a = f2a.dropna(subset=["District", "Upazila"]).copy()

for c in f2a.columns[2:]:
    f2a[c] = pd.to_numeric(f2a[c], errors="coerce").fillna(0)

print(f"DDM agriculture rows: {len(f2a)}")

# ============================================================
# CLEAN DISTRICT AND UPAZILA NAMES BEFORE MERGE
# ============================================================

def clean_admin_name(x):
    if pd.isna(x):
        return np.nan

    x = str(x)
    x = x.strip()
    x = " ".join(x.split())

    return x


for df in [f1, f2h, f2a]:
    df["District"] = df["District"].apply(clean_admin_name)
    df["Upazila"] = df["Upazila"].apply(clean_admin_name)

# ============================================================
# 7. MERGE SATELLITE AND DDM DATA
# ============================================================

m_house = pd.merge(f1, f2h, on=["District", "Upazila"], how="inner")
m_agri = pd.merge(f1, f2a, on=["District", "Upazila"], how="inner")

print(f"\nMerged house rows: {len(m_house)}")
print(f"Merged agriculture rows: {len(m_agri)}")

# ============================================================
# 8. CONFIGURATION
# ============================================================

FIGURE_GROUPS = [
    {
        
        "source": "house",
        "x_col": "Norm_Impact_House",
        "x_label": HOUSE_X_DISPLAY_LABEL,
        "layout": (2, 2),
        "figsize": (14, 10),
        "filename_base": "01_house_damage_count",
        "subplots": [
            {"y_col": "No_Brick", "label": LABEL_BRICK, "y_label": Y_LABEL_NO_BRICK, "color": "#2196F3"},
            {"y_col": "No_HalfBrick", "label": LABEL_HALFBRICK, "y_label": Y_LABEL_NO_HALFBRICK, "color": "#FF9800"},
            {"y_col": "No_Raw", "label": LABEL_RAW_HOUSE, "y_label": Y_LABEL_NO_RAW, "color": "#4CAF50"},
            {"y_col": "No_Total", "label": LABEL_TOTAL_HOUSE, "y_label": Y_LABEL_NO_TOTAL, "color": "#9C27B0"},
        ],
    },
    {
        
        "source": "house",
        "x_col": "Norm_Impact_House",
        "x_label": HOUSE_X_DISPLAY_LABEL,
        "layout": (2, 2),
        "figsize": (14, 10),
        "filename_base": "02_house_repair_amount",
        "subplots": [
            {"y_col": "Amt_Brick", "label": LABEL_AMT_BRICK, "y_label": Y_LABEL_AMT_BRICK, "color": "#2196F3"},
            {"y_col": "Amt_HalfBrick", "label": LABEL_AMT_HALFBRICK, "y_label": Y_LABEL_AMT_HALFBRICK, "color": "#FF9800"},
            {"y_col": "Amt_Raw", "label": LABEL_AMT_RAW, "y_label": Y_LABEL_AMT_RAW, "color": "#4CAF50"},
            {"y_col": "Amt_Total", "label": LABEL_AMT_TOTAL, "y_label": Y_LABEL_AMT_TOTAL, "color": "#9C27B0"},
        ],
    },
    {
       
        "source": "agri",
        "x_col": "Norm_Impact_fAPAR",
        "x_label": AGRI_X_DISPLAY_LABEL,
        "layout": (1, 3),
        "figsize": (18, 6),
        "filename_base": "03_agri_land_damage",
        "subplots": [
            {"y_col": "Fully_Land", "label": LABEL_FULLY_LAND, "y_label": Y_LABEL_FULLY_LAND, "color": "#1565C0"},
            {"y_col": "Partial_Land", "label": LABEL_PARTIAL_LAND, "y_label": Y_LABEL_PARTIAL_LAND, "color": "#00838F"},
            {"y_col": "Total_Loss_Land", "label": LABEL_TOTAL_LAND, "y_label": Y_LABEL_TOTAL_LOSS_LAND, "color": "#6A1B9A"},
        ],
    },
    {
       
        "source": "agri",
        "x_col": "Norm_Impact_fAPAR",
        "x_label": AGRI_X_DISPLAY_LABEL,
        "layout": (1, 3),
        "figsize": (18, 6),
        "filename_base": "04_agri_damage_amount",
        "subplots": [
            {"y_col": "Fully_Amt", "label": LABEL_FULLY_AMT, "y_label": Y_LABEL_FULLY_AMT, "color": "#B71C1C"},
            {"y_col": "Partial_Amt", "label": LABEL_PARTIAL_AMT, "y_label": Y_LABEL_PARTIAL_AMT, "color": "#E65100"},
            {"y_col": "Total_Loss_Amt", "label": LABEL_TOTAL_AMT, "y_label": Y_LABEL_TOTAL_LOSS_AMT, "color": "#1B5E20"},
        ],
    },
]

CATEGORICAL_CONFIG = []

for fig_cfg in FIGURE_GROUPS:
    for sp in fig_cfg["subplots"]:
        CATEGORICAL_CONFIG.append({
            "source": fig_cfg["source"],
            "x_col": fig_cfg["x_col"],
            "y_col": sp["y_col"],
            "label": sp["label"],
            "figure_group": fig_cfg["filename_base"],
        })

# ============================================================
# 8B. EXPANDED CLASS / BOXPLOT CONFIGURATION
# ============================================================
# This section controls categorical boxplot generation.
# Total boxplots produced:
# House = 8
# Agriculture = 6
# Total = 14

HOUSE_CLASS_BOXPLOT_CONFIG = [

    {
        "y_col": "No_Total",
        "y_label": Y_LABEL_NO_TOTAL,
        "label": "Forecasted Impact Catagory",
        "filename_base": "08_house_class_vs_no_total",
    },

    {
        "y_col": "Amt_Total",
        "y_label": Y_LABEL_AMT_TOTAL,
        "label": "Forecasted Impact Catagory",
        "filename_base": "12_house_class_vs_amt_total",
    },
]

AGRI_CLASS_BOXPLOT_CONFIG = [
 
    {
        "y_col": "Total_Loss_Land",
        "y_label": Y_LABEL_TOTAL_LOSS_LAND,
        "label": "Forecasted Impact Catagory",
        "filename_base": "17_agri_class_vs_total_loss_land",
    },
    {
        "y_col": "Total_Loss_Amt",
        "y_label": Y_LABEL_TOTAL_LOSS_AMT,
        "label": "Forecasted Impact Catagory",
        "filename_base": "18_agri_class_vs_total_loss_amt",
    },
]

CLASS_CONFIG = []

for item in HOUSE_CLASS_BOXPLOT_CONFIG:
    CLASS_CONFIG.append({
        "source": "house",
        "class_col": HOUSE_CLASS_COL,
        "impact_col": "Norm_Impact_House",
        "impact_label": HOUSE_X_DISPLAY_LABEL,
        "y_col": item["y_col"],
        "y_label": item["y_label"],
        "label": item["label"],
        "filename_base": item["filename_base"],
    })

for item in AGRI_CLASS_BOXPLOT_CONFIG:
    CLASS_CONFIG.append({
        "source": "agri",
        "class_col": AGRI_CLASS_COL,
        "impact_col": "Norm_Impact_fAPAR",
        "impact_label": AGRI_X_DISPLAY_LABEL,
        "y_col": item["y_col"],
        "y_label": item["y_label"],
        "label": item["label"],
        "filename_base": item["filename_base"],
    })

CLASS_ORDER = ["No Impact", "Low", "Moderate", "High"]
FORECAST_YES_CLASSES = ["Moderate", "High"]

# ============================================================
# 9. STATISTICAL FUNCTIONS
# ============================================================

def rmse(obs, pred):
    return np.sqrt(np.mean((obs - pred) ** 2))


def nse(obs, pred):
    denominator = np.sum((obs - np.mean(obs)) ** 2)
    if denominator == 0:
        return np.nan
    return 1 - np.sum((obs - pred) ** 2) / denominator


def safe_round(v, digits=4):
    try:
        v = float(v)
        if np.isnan(v) or np.isinf(v):
            return None
        return round(v, digits)
    except Exception:
        return None


def normalize_class_value(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    if s in ["no impact", "noimpact", "no_impact", "none", "no", "nil", "zero", "0"]:
        return "No Impact"

    if s in ["low", "l"]:
        return "Low"

    if s in ["moderate", "medium", "med", "m"]:
        return "Moderate"

    if s in ["high", "h"]:
        return "High"

    return str(x).strip().title()


def prepare_xy(df, x_col, y_col, log_y=False):
    t = df[["District", "Upazila", x_col, y_col]].copy()

    t[x_col] = pd.to_numeric(t[x_col], errors="coerce")
    t[y_col] = pd.to_numeric(t[y_col], errors="coerce")

    t = t.dropna(subset=[x_col, y_col])
    t = t[(t[x_col] > 0) & (t[y_col] > 0)]

    if log_y:
        t[y_col] = np.log1p(t[y_col])

    return t.reset_index(drop=True)


def run_ols(x, y):
    if len(x) < 3:
        return {
            "slope": np.nan,
            "intercept": np.nan,
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "r2": np.nan,
            "rmse": np.nan,
            "nse": np.nan,
            "spearman_rho": np.nan,
            "spearman_p": np.nan,
            "kendall_tau": np.nan,
            "kendall_p": np.nan,
            "skew_y": np.nan,
            "y_pred": None,
        }

    slope, intercept, pearson_r, pearson_p, std_err = stats.linregress(x, y)
    y_pred = intercept + slope * x

    spearman_rho, spearman_p = stats.spearmanr(x, y)
    kendall_tau, kendall_p = stats.kendalltau(x, y)

    return {
        "slope": slope,
        "intercept": intercept,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "r2": pearson_r ** 2,
        "rmse": rmse(y, y_pred),
        "nse": nse(y, y_pred),
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "kendall_tau": kendall_tau,
        "kendall_p": kendall_p,
        "skew_y": stats.skew(y, nan_policy="omit"),
        "y_pred": y_pred,
    }


def categorical_metrics(df, x_col, y_col, sat_threshold=0, obs_threshold=0):
    if x_col not in df.columns or y_col not in df.columns:
        return None

    t = df[["District", "Upazila", x_col, y_col]].copy()

    t[x_col] = pd.to_numeric(t[x_col], errors="coerce")
    t[y_col] = pd.to_numeric(t[y_col], errors="coerce")

    t = t.dropna(subset=[x_col, y_col]).reset_index(drop=True)

    if len(t) == 0:
        return None

    forecast_yes = t[x_col] > sat_threshold
    observed_yes = t[y_col] > obs_threshold

    hit = int(((forecast_yes == True) & (observed_yes == True)).sum())
    miss = int(((forecast_yes == False) & (observed_yes == True)).sum())
    false_alarm = int(((forecast_yes == True) & (observed_yes == False)).sum())
    correct_negative = int(((forecast_yes == False) & (observed_yes == False)).sum())

    pod = hit / (hit + miss) if (hit + miss) > 0 else np.nan
    far = false_alarm / (hit + false_alarm) if (hit + false_alarm) > 0 else np.nan
    csi = hit / (hit + miss + false_alarm) if (hit + miss + false_alarm) > 0 else np.nan
    bias = (hit + false_alarm) / (hit + miss) if (hit + miss) > 0 else np.nan
    accuracy = (hit + correct_negative) / len(t) if len(t) > 0 else np.nan

    return {
        "Samples Used": len(t),
        "Satellite Threshold": sat_threshold,
        "Observed Threshold": obs_threshold,
        "Hit": hit,
        "Miss": miss,
        "False Alarm": false_alarm,
        "Correct Negative": correct_negative,
        "POD": pod,
        "FAR": far,
        "CSI": csi,
        "Bias": bias,
        "Accuracy": accuracy,
    }


def optimize_noimpact_threshold(
    df,
    impact_col,
    damage_col,
    thresholds=None,
    obs_threshold=0,
    bias_min=0.80,
    bias_max=1.20,
    far_max=0.25
):
    if thresholds is None:
        thresholds = np.round(
            np.arange(THRESHOLD_MIN, THRESHOLD_MAX + THRESHOLD_STEP, THRESHOLD_STEP),
            2
        )

    if impact_col not in df.columns or damage_col not in df.columns:
        return pd.DataFrame(), np.nan

    t = df[["District", "Upazila", impact_col, damage_col]].copy()

    t[impact_col] = pd.to_numeric(t[impact_col], errors="coerce")
    t[damage_col] = pd.to_numeric(t[damage_col], errors="coerce")

    t = t.dropna(subset=[impact_col, damage_col]).reset_index(drop=True)

    if len(t) == 0:
        return pd.DataFrame(), np.nan

    observed_yes = t[damage_col] > obs_threshold

    rows = []

    for threshold in thresholds:
        forecast_yes = t[impact_col] > threshold

        hit = int(((forecast_yes == True) & (observed_yes == True)).sum())
        miss = int(((forecast_yes == False) & (observed_yes == True)).sum())
        false_alarm = int(((forecast_yes == True) & (observed_yes == False)).sum())
        correct_negative = int(((forecast_yes == False) & (observed_yes == False)).sum())

        pod = hit / (hit + miss) if (hit + miss) > 0 else np.nan
        far = false_alarm / (hit + false_alarm) if (hit + false_alarm) > 0 else np.nan
        csi = hit / (hit + miss + false_alarm) if (hit + miss + false_alarm) > 0 else np.nan
        bias = (hit + false_alarm) / (hit + miss) if (hit + miss) > 0 else np.nan
        accuracy = (hit + correct_negative) / len(t) if len(t) > 0 else np.nan

        bias_distance = abs(bias - 1) if not np.isnan(bias) else np.nan

        if np.isnan(csi) or np.isnan(far) or np.isnan(bias_distance):
            decision_score = np.nan
        else:
            decision_score = csi - far - bias_distance

        bias_ok = False if np.isnan(bias) else (bias >= bias_min and bias <= bias_max)
        far_ok = False if np.isnan(far) else (far <= far_max)

        rows.append({
            "Impact Column": impact_col,
            "Damage Column": damage_col,
            "Threshold": threshold,
            "Samples Used": len(t),
            "Hit": hit,
            "Miss": miss,
            "False Alarm": false_alarm,
            "Correct Negative": correct_negative,
            "POD": safe_round(pod),
            "FAR": safe_round(far),
            "CSI": safe_round(csi),
            "Bias": safe_round(bias),
            "Accuracy": safe_round(accuracy),
            "Bias Distance from 1": safe_round(bias_distance),
            "Bias Acceptable 0.80-1.20": "Yes" if bias_ok else "No",
            "FAR Acceptable <=0.25": "Yes" if far_ok else "No",
            DECISION_SCORE_NAME: safe_round(decision_score),
        })

    result_df = pd.DataFrame(rows)

    if len(result_df) == 0:
        return result_df, np.nan

    result_df["Candidate Pool"] = "Fallback_All"

    preferred_df = result_df[
        (result_df["Bias Acceptable 0.80-1.20"] == "Yes") &
        (result_df["FAR Acceptable <=0.25"] == "Yes")
    ].copy()

    if len(preferred_df) > 0:
        candidate_df = preferred_df.copy()
        candidate_pool_label = "Preferred_Bias_and_FAR_OK"
    else:
        candidate_df = result_df.copy()
        candidate_pool_label = "Fallback_All"

    candidate_df = candidate_df.dropna(subset=[DECISION_SCORE_NAME]).copy()

    if len(candidate_df) == 0:
        best_threshold = np.nan
        result_df["Best Threshold"] = "No"
        result_df["Candidate Pool"] = candidate_pool_label
        return result_df, best_threshold

    best_df = candidate_df.sort_values(
        by=[DECISION_SCORE_NAME, "CSI", "FAR", "Bias Distance from 1", "Threshold"],
        ascending=[False, False, True, True, True]
    ).head(1).copy()

    best_threshold = best_df["Threshold"].iloc[0]

    result_df["Candidate Pool"] = candidate_pool_label
    result_df["Best Threshold"] = result_df["Threshold"].apply(
        lambda x: "Yes" if x == best_threshold else "No"
    )

    return result_df, best_threshold


def mann_whitney_test(df, x_col, y_col, obs_threshold=0):
    if x_col not in df.columns or y_col not in df.columns:
        return None

    t = df[["District", "Upazila", x_col, y_col]].copy()

    t[x_col] = pd.to_numeric(t[x_col], errors="coerce")
    t[y_col] = pd.to_numeric(t[y_col], errors="coerce")

    t = t.dropna(subset=[x_col, y_col]).reset_index(drop=True)

    if len(t) == 0:
        return None

    damaged = t.loc[t[y_col] > obs_threshold, x_col].dropna().values
    no_damage = t.loc[t[y_col] <= obs_threshold, x_col].dropna().values

    if len(damaged) < 2 or len(no_damage) < 2:
        return {
            "Samples Used": len(t),
            "Observed Threshold": obs_threshold,
            "n Damaged": len(damaged),
            "n No Damage": len(no_damage),
            "Median Satellite Damaged": np.nan,
            "Median Satellite No Damage": np.nan,
            "Mean Satellite Damaged": np.nan,
            "Mean Satellite No Damage": np.nan,
            "U Statistic": np.nan,
            "p-value": np.nan,
            "Significant at 0.05": "Insufficient data",
            "Interpretation": "Insufficient damaged or non-damaged samples"
        }

    u_stat, p_value = stats.mannwhitneyu(
        damaged,
        no_damage,
        alternative="greater"
    )

    median_damaged = np.median(damaged)
    median_no_damage = np.median(no_damage)

    if p_value < 0.05 and median_damaged > median_no_damage:
        significant = "Yes"
        interpretation = "Satellite impact values are significantly higher in damaged upazilas"
    elif p_value < 0.05 and median_damaged <= median_no_damage:
        significant = "Yes, but direction issue"
        interpretation = "Statistically significant result, but median direction is not as expected"
    else:
        significant = "No"
        interpretation = "No statistically significant evidence that damaged upazilas have higher satellite impact values"

    return {
        "Samples Used": len(t),
        "Observed Threshold": obs_threshold,
        "n Damaged": len(damaged),
        "n No Damage": len(no_damage),
        "Median Satellite Damaged": median_damaged,
        "Median Satellite No Damage": median_no_damage,
        "Mean Satellite Damaged": np.mean(damaged),
        "Mean Satellite No Damage": np.mean(no_damage),
        "U Statistic": u_stat,
        "p-value": p_value,
        "Significant at 0.05": significant,
        "Interpretation": interpretation
    }

# ============================================================
# 10. OLS PLOTTING FUNCTION
# ============================================================

def plot_ols_group(fig_cfg, df, log_y=False):
    transformation = "log1p(Y)" if log_y else "Raw Y"
    rows, cols = fig_cfg["layout"]

    fig, axes = plt.subplots(rows, cols, figsize=fig_cfg["figsize"])

    if rows * cols == 1:
        ax_flat = np.array([axes])
    else:
        ax_flat = np.array(axes).flatten()

   

    summary_rows_group = []
    x_col = fig_cfg["x_col"]
    x_label = fig_cfg.get("x_label", x_col)

    for ax, sp in zip(ax_flat, fig_cfg["subplots"]):
        y_col = sp["y_col"]
        label = sp["label"]
        y_label = sp["y_label"]
        color = sp["color"]

        if x_col not in df.columns or y_col not in df.columns:
            ax.set_title(f"{label}\nColumn missing")
            ax.axis("off")
            continue

        t = prepare_xy(df=df, x_col=x_col, y_col=y_col, log_y=log_y)

        if len(t) < 3:
            ax.set_title(f"{label}\nInsufficient non-zero data")
            ax.grid(alpha=0.3)
            continue

        x = t[x_col].values.astype(float)
        y = t[y_col].values.astype(float)

        ols_result = run_ols(x, y)

        ax.scatter(
            x,
            y,
            alpha=0.65,
            s=45,
            color=color,
            label="Observed"
        )

        x_line = np.linspace(x.min(), x.max(), 300)
        y_line = ols_result["intercept"] + ols_result["slope"] * x_line

        ax.plot(
            x_line,
            y_line,
            "r-",
            linewidth=2,
            label=f"OLS R²={ols_result['r2']:.3f}"
        )

        ax.set_title(f"{label} (n={len(t)})", fontsize=10)
        ax.set_xlabel(x_label, fontsize=9)
        ax.set_ylabel(y_label, fontsize=9)

        ax.grid(alpha=0.3)
        ax.legend(loc='upper right', fontsize=7)

        text = (
            f"Pearson r={ols_result['pearson_r']:.3f}\n"
            f"Spearman ρ={ols_result['spearman_rho']:.3f}\n"
            f"p={ols_result['pearson_p']:.3g}\n"
            f"RMSE={ols_result['rmse']:.2g}\n"
            f"NSE={ols_result['nse']:.3f}"
        )

        ax.text(
            0.05,
            0.95,
            text,
            transform=ax.transAxes,
            va="top",
            fontsize=8,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
        )

        summary_rows_group.append({
            "Figure Group": fig_cfg["filename_base"],
            "Source": fig_cfg["source"],
            "Transformation": transformation,
            "X Variable": x_col,
            "X Display Label": x_label,
            "Y Variable": y_col,
            "Y Display Label": y_label,
            "Analysis Label": label,
            "Samples Used": len(t),

            "OLS Slope": safe_round(ols_result["slope"]),
            "OLS Intercept": safe_round(ols_result["intercept"]),
            "Pearson r": safe_round(ols_result["pearson_r"]),
            "Pearson p-value": safe_round(ols_result["pearson_p"]),
            "OLS R²": safe_round(ols_result["r2"]),
            "OLS RMSE": safe_round(ols_result["rmse"]),
            "OLS NSE": safe_round(ols_result["nse"]),

            "Spearman rho": safe_round(ols_result["spearman_rho"]),
            "Spearman p-value": safe_round(ols_result["spearman_p"]),
            "Kendall tau": safe_round(ols_result["kendall_tau"]),
            "Kendall p-value": safe_round(ols_result["kendall_p"]),
            "Skewness of Y": safe_round(ols_result["skew_y"]),
        })

    for extra_ax in ax_flat[len(fig_cfg["subplots"]):]:
        extra_ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.94])

    out_png = os.path.join(
        OUT_DIR,
        f"{fig_cfg['filename_base']}_{'log' if log_y else 'raw'}_ols_group.png"
    )

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved grouped OLS plot: {out_png}")

    return summary_rows_group

# ============================================================
# 11. CLASS DIAGNOSIS AND EXPANDED BOXPLOT FUNCTIONS
# ============================================================

def plot_class_boxplot(df, cfg):
    t = prepare_class_data(df, cfg["class_col"], cfg["impact_col"], cfg["y_col"])
    if t is None or len(t) == 0:
        print(f"Skipping boxplot: no data for {cfg['label']}")
        return None

    t = t[t[cfg["class_col"]].isin(CLASS_ORDER)].copy()
    if len(t) == 0:
        print(f"Skipping boxplot: no valid class data for {cfg['label']}")
        return None

    data, labels, positions, summary_info = [], [], [], []

    for i, cls in enumerate(CLASS_ORDER, start=1):
        g = t[t[cfg["class_col"]] == cls].copy()
        if len(g) == 0:
            continue
        vals = g["log1p_damage"].dropna().values
        if len(vals) == 0:
            continue
        data.append(vals)
        labels.append(cls)
        positions.append(i)
        summary_info.append({
            "class": cls,
            "n": len(g),
            "median": np.median(vals),
            "mean": np.mean(vals),
            "damaged_pct": 100 * g["Observed_Damage"].mean()
        })

    if len(data) < 2:
        print(f"Skipping boxplot: less than two valid classes for {cfg['label']}")
        return None

    fig, ax = plt.subplots(figsize=(11, 6.5))
    plt.subplots_adjust(right=0.76)

    ax.boxplot(
        data,
        positions=positions,
        labels=labels,
        showmeans=True,
        patch_artist=True,
        widths=0.55,
        meanprops={"marker":"^","markerfacecolor":"green","markeredgecolor":"green","markersize":5},
        medianprops={"color":"orange","linewidth":1.8},
        boxprops={"facecolor":"#2C7BB6","edgecolor":"black","linewidth":1.8,"alpha":0.25},
        whiskerprops={"linewidth":1.4,"color":"black"},
        capprops={"linewidth":1.4,"color":"black"}
    )

    rng = np.random.default_rng(42)
    point_colors = {"No Impact": "#BDBDBD", "Low": "#5DA5DA", "Moderate": "#FAA43A", "High": "#60BD68"}

    for pos, cls, vals in zip(positions, labels, data):
        jitter = rng.normal(loc=0, scale=0.05, size=len(vals))
        x_jittered = pos + jitter
        ax.scatter(
            x_jittered,
            vals,
            alpha=0.75,
            s=42,
            color=point_colors.get(cls, "gray"),
            edgecolor="black",
            linewidth=0.35,
            label="_nolegend_"
        )

    mean_handle = ax.scatter([], [], marker="^", color="green", s=35, label="Mean")
    median_handle, = ax.plot([], [], color="orange", linewidth=1.5, label="Median")
    ax.legend(handles=[mean_handle, median_handle],
              loc="upper left",
              bbox_to_anchor=(1.02, 1.00),
              fontsize=8,
              frameon=True,
              borderpad=0.4,
              handlelength=1.5,
              handletextpad=0.5)

    # Remove main title
    fig.text(
    0.5, 0.01,         # x=center, y=bottom
    cfg['label'],      # text
    ha='center',       # horizontal alignment
    va='bottom',       # vertical alignment
    fontsize=12,
    
)
    #ax.set_title("Forecasted Impact Catagory")

    # Y-axis label can remain or be blank
    ax.set_ylabel(cfg.get("y_label", cfg["y_col"]), fontsize=10)

    # Reduce x-axis tick font size
    ax.tick_params(axis='x', labelsize=1)

    ax.grid(alpha=0.3)
    ax.tick_params(axis="both", labelsize=9)

    y_positions = [0.78, 0.58, 0.38, 0.18]
    for info, y_pos in zip(summary_info, y_positions):
        txt = f"{info['class']}\nn={info['n']}\nMed={info['median']:.2f}\nMean={info['mean']:.2f}"
        ax.text(1.02, y_pos, txt, transform=ax.transAxes, ha="left", va="top", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="gray", alpha=0.95))

    out_png = os.path.join(OUT_DIR, f"{cfg['filename_base']}_boxplot_clean.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved clean class boxplot: {out_png}")

    return out_png

# ============================================================
# 11B. CLASS DATA PREPARATION AND DIAGNOSTICS (was missing)
# ============================================================

def prepare_class_data(df, class_col, impact_col, y_col):
    if class_col not in df.columns or y_col not in df.columns:
        return None

    t = df[["District", "Upazila", class_col, impact_col, y_col]].copy()

    t[class_col] = t[class_col].apply(normalize_class_value)
    t[impact_col] = pd.to_numeric(t[impact_col], errors="coerce")
    t[y_col] = pd.to_numeric(t[y_col], errors="coerce")

    t = t.dropna(subset=[class_col, y_col]).reset_index(drop=True)

    t["log1p_damage"] = np.log1p(t[y_col].clip(lower=0))
    t["Observed_Damage"] = (t[y_col] > 0).astype(int)

    return t


def class_diagnostics(df, cfg):
    class_col = cfg["class_col"]
    y_col = cfg["y_col"]
    impact_col = cfg["impact_col"]

    t = prepare_class_data(df, class_col, impact_col, y_col)

    if t is None or len(t) == 0:
        return pd.DataFrame()

    t = t[t[class_col].isin(CLASS_ORDER)].copy()

    if len(t) == 0:
        return pd.DataFrame()

    rows = []

    for cls in CLASS_ORDER:
        g = t[t[class_col] == cls]
        if len(g) == 0:
            continue

        rows.append({
            "Source": cfg["source"],
            "Analysis Label": cfg["label"],
            "Class Column": class_col,
            "Y Variable": y_col,
            "Y Display Label": cfg.get("y_label", y_col),
            "Class": cls,
            "n": len(g),
            "Median Damage": safe_round(g[y_col].median()),
            "Mean Damage": safe_round(g[y_col].mean()),
            "Median log1p Damage": safe_round(g["log1p_damage"].median()),
            "Mean log1p Damage": safe_round(g["log1p_damage"].mean()),
            "Damaged %": safe_round(100 * g["Observed_Damage"].mean()),
        })

    # Forecast Yes (Moderate/High) vs No (No Impact/Low) contingency stats
    t["Forecast_Yes"] = t[class_col].isin(FORECAST_YES_CLASSES)

    hit = int(((t["Forecast_Yes"]) & (t["Observed_Damage"] == 1)).sum())
    miss = int(((~t["Forecast_Yes"]) & (t["Observed_Damage"] == 1)).sum())
    false_alarm = int(((t["Forecast_Yes"]) & (t["Observed_Damage"] == 0)).sum())
    correct_negative = int(((~t["Forecast_Yes"]) & (t["Observed_Damage"] == 0)).sum())

    pod = hit / (hit + miss) if (hit + miss) > 0 else np.nan
    far = false_alarm / (hit + false_alarm) if (hit + false_alarm) > 0 else np.nan
    csi = hit / (hit + miss + false_alarm) if (hit + miss + false_alarm) > 0 else np.nan
    bias = (hit + false_alarm) / (hit + miss) if (hit + miss) > 0 else np.nan
    accuracy = (hit + correct_negative) / len(t) if len(t) > 0 else np.nan

    rows.append({
        "Source": cfg["source"],
        "Analysis Label": cfg["label"],
        "Class Column": class_col,
        "Y Variable": y_col,
        "Y Display Label": cfg.get("y_label", y_col),
        "Class": "Forecast Yes (Moderate+High) vs No (No Impact+Low)",
        "n": len(t),
        "Hit": hit,
        "Miss": miss,
        "False Alarm": false_alarm,
        "Correct Negative": correct_negative,
        "POD": safe_round(pod),
        "FAR": safe_round(far),
        "CSI": safe_round(csi),
        "Bias": safe_round(bias),
        "Accuracy": safe_round(accuracy),
    })

    return pd.DataFrame(rows)
# ============================================================
# 12. RUN OLS
# ============================================================

summary_rows = []

print("\nRunning grouped OLS exploratory analysis...")

for fig_cfg in FIGURE_GROUPS:
    df = m_house if fig_cfg["source"] == "house" else m_agri

    for log_y in [False, True]:
        rows = plot_ols_group(
            fig_cfg=fig_cfg,
            df=df,
            log_y=log_y
        )

        summary_rows.extend(rows)

df_summary = pd.DataFrame(summary_rows)

# ============================================================
# 13. RUN CATEGORICAL MATRIX
# ============================================================

categorical_rows = []

print("\nRunning categorical matrix validation...")

SAT_THRESHOLD = 0
OBS_THRESHOLD = 0

for cfg in CATEGORICAL_CONFIG:
    df = m_house if cfg["source"] == "house" else m_agri

    result = categorical_metrics(
        df=df,
        x_col=cfg["x_col"],
        y_col=cfg["y_col"],
        sat_threshold=SAT_THRESHOLD,
        obs_threshold=OBS_THRESHOLD
    )

    if result is None:
        continue

    categorical_rows.append({
        "Figure Group": cfg["figure_group"],
        "Source": cfg["source"],
        "Analysis Label": cfg["label"],
        "X Variable": cfg["x_col"],
        "Y Variable": cfg["y_col"],

        "Samples Used": result["Samples Used"],
        "Satellite Threshold": result["Satellite Threshold"],
        "Observed Threshold": result["Observed Threshold"],

        "Hit": result["Hit"],
        "Miss": result["Miss"],
        "False Alarm": result["False Alarm"],
        "Correct Negative": result["Correct Negative"],

        "POD": safe_round(result["POD"]),
        "FAR": safe_round(result["FAR"]),
        "CSI": safe_round(result["CSI"]),
        "Bias": safe_round(result["Bias"]),
        "Accuracy": safe_round(result["Accuracy"]),
    })

df_categorical = pd.DataFrame(categorical_rows)

print(f"Categorical matrix rows: {len(df_categorical)}")

# ============================================================
# 14. RUN MANN-WHITNEY U TEST
# ============================================================

mann_whitney_rows = []

print("\nRunning Mann-Whitney U test...")

for cfg in CATEGORICAL_CONFIG:
    df = m_house if cfg["source"] == "house" else m_agri

    result = mann_whitney_test(
        df=df,
        x_col=cfg["x_col"],
        y_col=cfg["y_col"],
        obs_threshold=OBS_THRESHOLD
    )

    if result is None:
        continue

    mann_whitney_rows.append({
        "Figure Group": cfg["figure_group"],
        "Source": cfg["source"],
        "Analysis Label": cfg["label"],
        "X Variable": cfg["x_col"],
        "Y Variable": cfg["y_col"],

        "Samples Used": result["Samples Used"],
        "Observed Threshold": result["Observed Threshold"],
        "n Damaged": result["n Damaged"],
        "n No Damage": result["n No Damage"],

        "Median Satellite Damaged": safe_round(result["Median Satellite Damaged"]),
        "Median Satellite No Damage": safe_round(result["Median Satellite No Damage"]),
        "Mean Satellite Damaged": safe_round(result["Mean Satellite Damaged"]),
        "Mean Satellite No Damage": safe_round(result["Mean Satellite No Damage"]),

        "U Statistic": safe_round(result["U Statistic"]),
        "p-value": safe_round(result["p-value"]),
        "Significant at 0.05": result["Significant at 0.05"],
        "Interpretation": result["Interpretation"],
    })

df_mannwhitney = pd.DataFrame(mann_whitney_rows)

print(f"Mann-Whitney U test rows: {len(df_mannwhitney)}")

# ============================================================
# 15. RUN CLASS DIAGNOSIS AND 14 BOXPLOTS
# ============================================================

print("\nRunning expanded class diagnosis and boxplot generation...")

diagnostic_frames = []
boxplot_files = []

for cfg in CLASS_CONFIG:
    df = m_house if cfg["source"] == "house" else m_agri

    if cfg["class_col"] not in df.columns:
        print(f"Skipping {cfg['label']}: class column missing: {cfg['class_col']}")
        continue

    if cfg["y_col"] not in df.columns:
        print(f"Skipping {cfg['label']}: damage column missing: {cfg['y_col']}")
        continue

    diag_df = class_diagnostics(df, cfg)

    if len(diag_df) > 0:
        diagnostic_frames.append(diag_df)

    out_plot = plot_class_boxplot(df, cfg)

    if out_plot is not None:
        boxplot_files.append(out_plot)

df_class_diagnostics = (
    pd.concat(diagnostic_frames, ignore_index=True)
    if len(diagnostic_frames) > 0 else pd.DataFrame()
)

print(f"Class diagnosis rows: {len(df_class_diagnostics)}")
print(f"Total class boxplots saved: {len(boxplot_files)}")

# ============================================================
# 15B. BALANCED HOUSE THRESHOLD OPTIMIZATION
# ============================================================

print("\nRunning balanced threshold optimization for No Impact / Low separation...")

house_threshold_results = []

HOUSE_THRESHOLD_TESTS = [
    {
        "impact_col": "Norm_Impact_House",
        "damage_col": "No_Total",
        "label": "House Impact Threshold vs Total Damaged House"
    },
    {
        "impact_col": "Norm_Impact_House",
        "damage_col": "Amt_Total",
        "label": "House Impact Threshold vs Total Repair Amount"
    }
]

thresholds_to_test = np.round(
    np.arange(THRESHOLD_MIN, THRESHOLD_MAX + THRESHOLD_STEP, THRESHOLD_STEP),
    2
)

for test in HOUSE_THRESHOLD_TESTS:
    if test["impact_col"] not in m_house.columns:
        print(f"Skipping threshold test: missing {test['impact_col']}")
        continue

    if test["damage_col"] not in m_house.columns:
        print(f"Skipping threshold test: missing {test['damage_col']}")
        continue

    result_df, best_threshold = optimize_noimpact_threshold(
        df=m_house,
        impact_col=test["impact_col"],
        damage_col=test["damage_col"],
        thresholds=thresholds_to_test,
        obs_threshold=0,
        bias_min=BIAS_MIN_ACCEPTABLE,
        bias_max=BIAS_MAX_ACCEPTABLE,
        far_max=FAR_MAX_ACCEPTABLE
    )

    if len(result_df) == 0:
        continue

    result_df.insert(0, "Analysis Label", test["label"])
    result_df.insert(1, "Recommended Threshold", best_threshold)
    result_df.insert(2, "Decision Rule", DECISION_RULE_DESCRIPTION)

    house_threshold_results.append(result_df)

    print(f"{test['label']}: Recommended balanced threshold = {best_threshold}")

df_house_threshold = (
    pd.concat(house_threshold_results, ignore_index=True)
    if len(house_threshold_results) > 0 else pd.DataFrame()
)

print(f"Threshold optimization rows: {len(df_house_threshold)}")

# ============================================================
# 16. EXPORT FIVE SHEETS
# ============================================================

print("\nWriting Excel output...")

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    df_summary.to_excel(
        writer,
        sheet_name="OLS Summary",
        index=False,
        startrow=2
    )

    df_categorical.to_excel(
        writer,
        sheet_name="Categorical Matrix",
        index=False,
        startrow=2
    )

    df_mannwhitney.to_excel(
        writer,
        sheet_name="Mann Whitney U Test",
        index=False,
        startrow=2
    )

    df_class_diagnostics.to_excel(
        writer,
        sheet_name="Class Diagnosis",
        index=False,
        startrow=2
    )

    df_house_threshold.to_excel(
        writer,
        sheet_name="Threshold Optimization",
        index=False,
        startrow=2
    )

# ============================================================
# 17. FORMAT EXCEL
# ============================================================

wb = load_workbook(OUT_EXCEL)

HDR_FILL = PatternFill("solid", fgColor="1F3864")
HDR_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=9)
TITLE_FONT = Font(bold=True, size=12, name="Arial", color="1F3864")

thin = Side(style="thin", color="CCCCCC")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT = Alignment(horizontal="left", vertical="center", wrap_text=True)

for sheet in wb.sheetnames:
    ws = wb[sheet]

    ws.cell(row=1, column=1).value = sheet
    ws.cell(row=1, column=1).font = TITLE_FONT
    ws.cell(row=1, column=1).alignment = LEFT

    ws.cell(row=2, column=1).value = (
        "Impact-based forecast validation. "
        "OLS uses non-zero satellite-ground pairs only. "
        "Categorical Matrix, Mann Whitney U Test, Class Diagnosis, and Threshold Optimization use zero and non-zero values. "
        "fAPAR/agriculture category uses Excel column X. "
        "House category uses Excel column AG. "
        "Expanded class boxplots include all requested DDM house columns C-J and agriculture columns C-H. "
        "House class supports No Impact, Low, Moderate, and High. "
        "Balanced Threshold Optimization prefers Bias between 0.80 and 1.20, FAR <= 0.25, "
        "and selects the highest Decision Score = CSI - FAR - abs(Bias - 1). "
        "Log transformation is used internally for boxplots and log plots, but axis labels are displayed without log1p."
    )

    ws.cell(row=2, column=1).font = Font(
        italic=True,
        size=9,
        name="Arial",
        color="444444"
    )

    ws.cell(row=2, column=1).alignment = LEFT

    if ws.max_row >= 3:
        for col_idx in range(1, ws.max_column + 1):
            cell = ws.cell(row=3, column=col_idx)
            cell.fill = HDR_FILL
            cell.font = HDR_FONT
            cell.alignment = CENTER
            cell.border = BORDER

        for row_idx in range(4, ws.max_row + 1):
            for col_idx in range(1, ws.max_column + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.border = BORDER
                cell.alignment = LEFT
                cell.font = Font(name="Arial", size=9)

        for col_idx in range(1, ws.max_column + 1):
            col_letter = get_column_letter(col_idx)
            header = str(ws.cell(row=3, column=col_idx).value or "")

            max_data_len = max(
                [
                    len(str(ws.cell(row=r, column=col_idx).value or ""))
                    for r in range(4, ws.max_row + 1)
                ],
                default=8
            )

            ws.column_dimensions[col_letter].width = min(
                max(len(header), max_data_len) + 3,
                42
            )

        ws.freeze_panes = "C4"

wb.save(OUT_EXCEL)

# ============================================================
# 18. FINAL MESSAGE
# ============================================================

print("\n============================================================")
print("VALIDATION WITH EXPANDED CATEGORICAL BOXPLOTS COMPLETED")
print("============================================================")
print(f"Output folder: {OUT_DIR}")
print(f"Excel file: {OUT_EXCEL}")
print(f"Total OLS summary rows: {len(df_summary)}")
print(f"Total categorical matrix rows: {len(df_categorical)}")
print(f"Total Mann-Whitney U test rows: {len(df_mannwhitney)}")
print(f"Total class diagnosis rows: {len(df_class_diagnostics)}")
print(f"Total threshold optimization rows: {len(df_house_threshold)}")
print(f"Total class boxplots saved: {len(boxplot_files)}")
print("Grouped OLS PNG figures were saved.")
print("Expanded clean class boxplot PNG figures were saved.")
print("Expected class boxplots:")
print("  House: 8")
print("    1. Brick-built House")
print("    2. Half Brick-built House")
print("    3. Kacha/Raw House")
print("    4. Total Damaged House")
print("    5. Amount Spent on Brick-built House Repair")
print("    6. Amount Spent on Half Brick-built House Repair")
print("    7. Amount Spent on Kacha/Raw House Repair")
print("    8. Total Amount Spent on Repairs")
print("  Agriculture: 6")
print("    1. Fully Damaged Total Land Area")
print("    2. Fully Damaged Total Amount")
print("    3. Partially Damaged Total Land Area")
print("    4. Partially Damaged Total Amount")
print("    5. Total Loss Land Area")
print("    6. Total Loss Amount")
print("Excel file contains five sheets:")
print("  1. OLS Summary")
print("  2. Categorical Matrix")
print("  3. Mann Whitney U Test")
print("  4. Class Diagnosis")
print("  5. Threshold Optimization")
print("OLS excludes zero pairs.")
print("Categorical, Mann-Whitney, class diagnosis, and threshold optimization analyses keep zero and non-zero values.")
print("House class supports No Impact, Low, Moderate, and High.")
print("Forecast Yes classes are Moderate and High.")
print("Forecast No classes are No Impact and Low.")
print("Balanced threshold selection rule:")
print(f"  Bias acceptable range: {BIAS_MIN_ACCEPTABLE} to {BIAS_MAX_ACCEPTABLE}")
print(f"  FAR acceptable maximum: {FAR_MAX_ACCEPTABLE}")
print("  Decision Score = CSI - FAR - abs(Bias - 1)")
print("Axis labels are displayed without log1p, but log transformation is still applied internally where required.")
print(f"fAPAR/agriculture class column used: {AGRI_CLASS_COL}")
print(f"House class column used: {HOUSE_CLASS_COL}")
print("============================================================")

Loading satellite data...
Satellite rows: 544
Agriculture/fAPAR class column from Excel X: Classification.1
House class column from Excel AG: Classification.4

Unique fAPAR/agriculture classes:
['Moderate' 'High' 'Low' 'No Impact']

Unique house classes:
['High' 'No Impact' 'Moderate' 'Low']

Loading DDM house damage data...
DDM house rows: 154

Loading DDM agriculture damage data...
DDM agriculture rows: 154

Merged house rows: 125
Merged agriculture rows: 125

Running grouped OLS exploratory analysis...
Saved grouped OLS plot: C:\Users\Ibrahim\Desktop\Github_test\results\01_house_damage_count_raw_ols_group.png
Saved grouped OLS plot: C:\Users\Ibrahim\Desktop\Github_test\results\01_house_damage_count_log_ols_group.png
Saved grouped OLS plot: C:\Users\Ibrahim\Desktop\Github_test\results\02_house_repair_amount_raw_ols_group.png
Saved grouped OLS plot: C:\Users\Ibrahim\Desktop\Github_test\results\02_house_repair_amount_log_ols_group.png
Saved grouped OLS plot: C:\Users\Ibrahim\Desktop\Gi

In [2]:
# ============================================================
# OPTIMAL PERCENTILE THRESHOLD VISUALISATION
# Two figures: one for Norm_Impact_House, one for Norm_Impact_fAPAR
# Each figure: 2 panels
#   Left  — POD / FAR / CSI / Accuracy + Decision Score overlay
#   Right — Satellite No-Impact Rate vs Ground Truth coverage
# Best percentile selected by Decision Score = CSI - FAR - |Bias - 1|
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from openpyxl.utils import column_index_from_string
import os
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────
# FILE PATHS  — update for your machine
# ──────────────────────────────────────────────────────────────
FILE1 = r"C:\Users\Ibrahim\Desktop\Github_test\Forecasted_Impact.xlsx"
FILE2 = r"C:\Users\Ibrahim\Desktop\Github_test\Remal_ddm.xlsx"
OUT_DIR = r"C:\Users\Ibrahim\Desktop\Github_test\results\No_Impact_Threshold_Optimization"

F1_SHEET       = "Remal"
F2_HOUSE_SHEET = "House"
F2_AGRI_SHEET  = "Agriculture"

os.makedirs(OUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────
# SETTINGS
# ──────────────────────────────────────────────────────────────
PERCENTILES = list(range(5, 55, 5))   # 5, 10, 15 … 50

# ──────────────────────────────────────────────────────────────
# LOAD & CLEAN
# ──────────────────────────────────────────────────────────────
def clean_admin(x):
    if pd.isna(x): return np.nan
    return " ".join(str(x).strip().split())

f1 = pd.read_excel(FILE1, sheet_name=F1_SHEET)
f1.columns = [str(c).strip() for c in f1.columns]
f1 = (f1.dropna(subset=["District", "Upazila"])
        .drop_duplicates(subset=["District", "Upazila"], keep="first")
        .reset_index(drop=True))

raw_house = pd.read_excel(FILE2, sheet_name=F2_HOUSE_SHEET, header=None)
f2h = raw_house.iloc[2:, :10].copy().reset_index(drop=True)
f2h.columns = ["District","Upazila","No_Brick","No_HalfBrick","No_Raw","No_Total",
               "Amt_Brick","Amt_HalfBrick","Amt_Raw","Amt_Total"]
f2h["District"] = f2h["District"].ffill()
f2h = f2h.dropna(subset=["District","Upazila"])
for c in f2h.columns[2:]:
    f2h[c] = pd.to_numeric(f2h[c], errors="coerce").fillna(0)

raw_agri = pd.read_excel(FILE2, sheet_name=F2_AGRI_SHEET, header=None)
f2a = raw_agri.iloc[2:].copy().reset_index(drop=True)
if f2a.shape[1] >= 8:
    f2a = f2a.iloc[:, :8].copy()
    f2a.columns = ["District","Upazila","Fully_Land","Fully_Amt",
                   "Partial_Land","Partial_Amt","Total_Loss_Land","Total_Loss_Amt"]
else:
    f2a = f2a.iloc[:, :6].copy()
    f2a.columns = ["District","Upazila","Fully_Land","Fully_Amt","Partial_Land","Partial_Amt"]
    f2a["Total_Loss_Land"] = (pd.to_numeric(f2a["Fully_Land"],  errors="coerce").fillna(0) +
                               pd.to_numeric(f2a["Partial_Land"],errors="coerce").fillna(0))
    f2a["Total_Loss_Amt"]  = (pd.to_numeric(f2a["Fully_Amt"],   errors="coerce").fillna(0) +
                               pd.to_numeric(f2a["Partial_Amt"], errors="coerce").fillna(0))
f2a["District"] = f2a["District"].ffill()
f2a = f2a.dropna(subset=["District","Upazila"])
for c in f2a.columns[2:]:
    f2a[c] = pd.to_numeric(f2a[c], errors="coerce").fillna(0)

for df in [f1, f2h, f2a]:
    df["District"] = df["District"].apply(clean_admin)
    df["Upazila"]  = df["Upazila"].apply(clean_admin)

m_house = pd.merge(f1, f2h, on=["District","Upazila"], how="inner")
m_agri  = pd.merge(f1, f2a, on=["District","Upazila"], how="inner")

# ──────────────────────────────────────────────────────────────
# METRIC CALCULATION
# ──────────────────────────────────────────────────────────────
def compute_metrics(df, impact_col, damage_col):
    """
    Returns a DataFrame with one row per percentile containing all metrics,
    and the index of the best row (max Decision Score).
    """
    t = df[[impact_col, damage_col]].copy()
    t[impact_col] = pd.to_numeric(t[impact_col], errors="coerce")
    t[damage_col] = pd.to_numeric(t[damage_col], errors="coerce")
    t = t.dropna().reset_index(drop=True)

    obs  = t[damage_col] > 0
    gnir = (~obs).mean()                    # ground no-impact rate
    vals = t[impact_col].values

    rows = []
    for pct in PERCENTILES:
        th  = float(np.percentile(vals, pct))
        fy  = t[impact_col] > th

        hit  = int(( fy &  obs).sum())
        miss = int((~fy &  obs).sum())
        fa   = int(( fy & ~obs).sum())
        cn   = int((~fy & ~obs).sum())

        pod  = hit/(hit+miss)    if (hit+miss)    > 0 else np.nan
        far  = fa /(hit+fa)      if (hit+fa)      > 0 else np.nan
        csi  = hit/(hit+miss+fa) if (hit+miss+fa) > 0 else np.nan
        bias = (hit+fa)/(hit+miss) if (hit+miss)  > 0 else np.nan
        acc  = (hit+cn)/len(t)
        ds   = (csi - far - abs(bias-1)
                if not any(np.isnan([csi, far, bias])) else np.nan)
        sat_nir = (~fy).mean()

        rows.append(dict(
            pct=pct, th=th,
            pod=pod, far=far, csi=csi, bias=bias, acc=acc,
            ds=ds, sat_nir=sat_nir, gnir=gnir
        ))

    df_m = pd.DataFrame(rows)
    best_idx = int(df_m["ds"].idxmax())
    return df_m, best_idx

# ──────────────────────────────────────────────────────────────
# PLOTTING FUNCTION  — one figure per impact variable
# ──────────────────────────────────────────────────────────────
def plot_optimal_percentile(df_m, best_idx,
                            impact_label, damage_label,
                            out_path, variable_name, cyclone_name="Sitrang"):
    # variable_name: short label used only in the Panel A title,
    # e.g. "House" or "Agriculture" — everything else in the
    # function is unchanged.

    best      = df_m.loc[best_idx]
    best_pct  = int(best["pct"])
    best_th   = float(best["th"])
    best_ds   = float(best["ds"])
    gnir      = float(best["gnir"])

    pcts = df_m["pct"].values

    # ── colour palette ─────────────────────────────────────────
    C_POD  = "#1565C0"
    C_FAR  = "#C62828"
    C_CSI  = "#2E7D32"
    C_ACC  = "#6A1B9A"
    C_DS   = "#E65100"
    C_NIR  = "#0D47A1"
    C_GT   = "#F57F17"
    C_BEST = "#000000"

    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=(15, 6.5),
        gridspec_kw={"width_ratios": [1.6, 1]}
    )

    fig.suptitle(
        f"Cyclone {cyclone_name}  |  Optimal No-Impact Threshold: {impact_label}\n"
        f"Best percentile = P{best_pct}  "
        f"(Threshold ≤ {best_th:.4f} on 0–1 normalised scale)  "
        f"selected by Decision Score = CSI − FAR − |Bias − 1|",
        fontsize=12, fontweight="bold", y=1.02
    )

    # ══════════════════════════════════════════════════════════
    # LEFT PANEL — metrics + decision score
    # ══════════════════════════════════════════════════════════
    ax2 = ax_left.twinx()          # right y-axis for Decision Score

    # ── metric lines ───────────────────────────────────────────
    lw = 2.0
    l_pod, = ax_left.plot(pcts, df_m["pod"], color=C_POD, lw=lw,
                          marker="o", ms=7, zorder=4, label="POD")
    l_far, = ax_left.plot(pcts, df_m["far"], color=C_FAR, lw=lw,
                          marker="s", ms=7, zorder=4, label="FAR")
    l_csi, = ax_left.plot(pcts, df_m["csi"], color=C_CSI, lw=lw,
                          marker="^", ms=7, zorder=4, label="CSI")
    l_acc, = ax_left.plot(pcts, df_m["acc"], color=C_ACC, lw=lw,
                          marker="D", ms=7, zorder=4, linestyle="--",
                          label="Accuracy")

    # ── Decision Score on right y-axis ─────────────────────────
    ax2.fill_between(pcts, df_m["ds"], 0,
                     where=df_m["ds"] >= 0,
                     alpha=0.12, color=C_DS, interpolate=True)
    ax2.fill_between(pcts, df_m["ds"], 0,
                     where=df_m["ds"] < 0,
                     alpha=0.08, color="grey", interpolate=True)
    l_ds, = ax2.plot(pcts, df_m["ds"], color=C_DS, lw=2.4,
                     marker="P", ms=9, zorder=5,
                     label="Decision Score\n(CSI − FAR − |Bias−1|)")
    ax2.axhline(0, color=C_DS, lw=0.9, ls="--", alpha=0.45)

    # ── star markers at best percentile ────────────────────────
    for ax_obj, series, col in [
        (ax_left, df_m["pod"], C_POD),
        (ax_left, df_m["far"], C_FAR),
        (ax_left, df_m["csi"], C_CSI),
        (ax_left, df_m["acc"], C_ACC),
    ]:
        ax_obj.scatter(best_pct, series[best_idx],
                       s=200, color=col, marker="*",
                       edgecolors="black", linewidths=0.7, zorder=10)
    ax2.scatter(best_pct, best_ds,
                s=280, color=C_DS, marker="*",
                edgecolors="black", linewidths=0.8, zorder=10)

    # ── best percentile vertical line ──────────────────────────
    ax_left.axvline(best_pct, color=C_BEST, lw=2.0, ls="--", zorder=6)
    ax2.axvline(best_pct,     color=C_BEST, lw=2.0, ls="--", zorder=6)

    # ── annotation box ─────────────────────────────────────────
    annot = (
        f"  Optimal: P{best_pct}\n"
        f"  Threshold = {best_th:.4f}\n"
        f"  ─────────────────\n"
        f"  CSI      = {best['csi']:.3f}\n"
        f"  POD      = {best['pod']:.3f}\n"
        f"  FAR      = {best['far']:.3f}\n"
        f"  Bias     = {best['bias']:.3f}\n"
        f"  Dec.Score= {best_ds:.3f}"
    )
    # position annotation to the right of best_pct if space, else left
    x_offset = 3 if best_pct <= 30 else -3
    ha_ann   = "left" if best_pct <= 30 else "right"
    ax_left.annotate(
        annot,
        xy=(best_pct, best["csi"]),
        xytext=(best_pct + x_offset, 0.55),
        fontsize=8.5, ha=ha_ann,
        arrowprops=dict(arrowstyle="->", color="black", lw=1.3),
        bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                  edgecolor="black", lw=1.3, alpha=0.96),
        zorder=20
    )

    # ── axes formatting ────────────────────────────────────────
    ax_left.set_xlim(3, 52)
    ax_left.set_ylim(-0.05, 1.15)
    ax_left.xaxis.set_major_locator(mticker.MultipleLocator(5))
    ax_left.set_xlabel("Percentile used as No-Impact cut-off", fontsize=11)
    ax_left.set_ylabel("POD / FAR / CSI / Accuracy", fontsize=11)
    ax_left.set_title(
        f"On 24hr Lead-Time Comparison of Verification Metrics, Decision Score, Percentiles of Forecasted Impact of {variable_name}",
        fontsize=9.5, fontweight="bold"
    )
    ax_left.grid(alpha=0.25)

    ds_range = df_m["ds"].max() - df_m["ds"].min()
    ax2.set_ylim(df_m["ds"].min() - ds_range * 0.3,
                 df_m["ds"].max() + ds_range * 0.5)
    ax2.set_ylabel("Decision Score  (right axis)", fontsize=11, color=C_DS)
    ax2.tick_params(axis="y", colors=C_DS)
    ax2.spines["right"].set_edgecolor(C_DS)

    # ── legend (combined from both axes) ───────────────────────
    handles = [l_pod, l_far, l_csi, l_acc, l_ds,
               Line2D([0],[0], color=C_BEST, lw=2.0, ls="--",
                      label=f"Optimal: P{best_pct}")]
    ax_left.legend(handles=handles, loc="upper right",
                   fontsize=8.5, framealpha=0.95, ncol=2)

    # ── per-bar DS ticks below the zero line ───────────────────
    for _, row in df_m.iterrows():
        clr = C_DS if int(row["pct"]) == best_pct else "#FFCC80"
        ax2.bar(row["pct"], row["ds"], width=2.2,
                alpha=0.18, color=clr, zorder=2)

    # ══════════════════════════════════════════════════════════
    # RIGHT PANEL — satellite No-Impact rate vs ground truth
    # ══════════════════════════════════════════════════════════
    ax_right.plot(pcts, df_m["sat_nir"] * 100, color=C_NIR,
                  marker="o", ms=7, lw=2.0,
                  label="Forecasted No-Impact Rate (%)")

    ax_right.axhline(gnir * 100, color=C_GT, ls="--", lw=2.0,
                     label=f"DDM Observed No-Impact ({gnir*100:.1f}%)")
    ax_right.axhspan((gnir - 0.15) * 100, (gnir + 0.15) * 100,
                     alpha=0.10, color=C_GT,
                     label="±15% tolerance band")

    # star at best
    ax_right.scatter(best_pct, df_m.loc[best_idx, "sat_nir"] * 100,
                     s=220, color=C_NIR, marker="*",
                     edgecolors="black", linewidths=0.7, zorder=10)
    ax_right.axvline(best_pct, color=C_BEST, lw=2.0, ls="--",
                     label=f"Optimal: P{best_pct}")

    # label every point with its sat_nir %
    for _, row in df_m.iterrows():
        ax_right.text(
            row["pct"], row["sat_nir"] * 100 + 1.2,
            f"{row['sat_nir']*100:.0f}%",
            ha="center", va="bottom", fontsize=7.5, color=C_NIR
        )

    ax_right.set_xlim(3, 52)
    ax_right.set_ylim(-5, 75)
    ax_right.xaxis.set_major_locator(mticker.MultipleLocator(5))
    ax_right.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax_right.set_xlabel("Percentile used as No-Impact cut-off", fontsize=11)
    ax_right.set_ylabel("% of Upazilas classified as No Impact", fontsize=11)
    ax_right.set_title(
        "On 24hr Lead-Time No-Impact Coverage Rate vs DDM Observed",
        fontsize=9.5, fontweight="bold"
    )
    ax_right.grid(alpha=0.25)
    ax_right.legend(fontsize=8.5, loc="upper left", framealpha=0.95)

    # explanatory footnote inside right panel
    ax_right.text(
        0.99, 0.03,
        f"At P{best_pct}: {df_m.loc[best_idx,'sat_nir']*100:.0f}% of upazilas\n"
        f"called No Impact  (DDM Observed: {gnir*100:.0f}%)\n"
        f"Threshold = {best_th:.4f} on 0–1 scale",
        transform=ax_right.transAxes,
        ha="right", va="bottom", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="grey", alpha=0.92)
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_path}")


# ──────────────────────────────────────────────────────────────
# RUN — House
# ──────────────────────────────────────────────────────────────
print("Processing Norm_Impact_House vs No_Total …")
df_m_house, best_idx_house = compute_metrics(m_house, "Norm_Impact_House", "No_Total")

plot_optimal_percentile(
    df_m       = df_m_house,
    best_idx   = best_idx_house,
    impact_label = "Normalized Impact of House  (Norm_Impact_House)",
    damage_label = "No. of Damaged Houses",
    out_path   = os.path.join(OUT_DIR, "House_optimal_percentile_threshold.png"),
    variable_name = "House",
)

# ──────────────────────────────────────────────────────────────
# RUN — fAPAR
# ──────────────────────────────────────────────────────────────
print("Processing Norm_Impact_fAPAR vs Total_Loss_Land …")
df_m_agri, best_idx_agri = compute_metrics(m_agri, "Norm_Impact_fAPAR", "Total_Loss_Land")

plot_optimal_percentile(
    df_m       = df_m_agri,
    best_idx   = best_idx_agri,
    impact_label = "Normalized Impact of fAPAR  (Norm_Impact_fAPAR)",
    damage_label = "Total Agricultural Land Loss (ha)",
    out_path   = os.path.join(OUT_DIR, "fAPAR_optimal_percentile_threshold.png"),
    variable_name = "Agriculture",
)

print("\nDone. Two figures saved to:", OUT_DIR)

Processing Norm_Impact_House vs No_Total …
Saved: C:\Users\Ibrahim\Desktop\Github_test\results\No_Impact_Threshold_Optimization\House_optimal_percentile_threshold.png
Processing Norm_Impact_fAPAR vs Total_Loss_Land …
Saved: C:\Users\Ibrahim\Desktop\Github_test\results\No_Impact_Threshold_Optimization\fAPAR_optimal_percentile_threshold.png

Done. Two figures saved to: C:\Users\Ibrahim\Desktop\Github_test\results\No_Impact_Threshold_Optimization


In [3]:
# ============================================================
# IMPACT-BASED FORECAST VALIDATION — POLYGON SHAPEFILE EXPORT
# Cyclone Remal
#
# FIX: Shapefile join uses ADM3_PCODE only — not upazila name.
#      13 upazila names repeat across districts (e.g. Kaliganj
#      exists in Gazipur, Jhenaidah, Lalmonirhat, Satkhira).
#      Joining by name alone would assign the wrong polygon.
#      ADM3_PCODE is unique per upazila and solves this.
#
# MAPS:
#   Map1 - House: col AF (Norm_Impact_House) vs col F (No_Total)
#   Map2 - House: col AF (Norm_Impact_House) vs col J (Amt_Total)
#   Map3 - Agri : col W  (Norm_Impact_fAPAR) vs col G (Total_Loss_Land)
#   Map4 - Agri : col W  (Norm_Impact_fAPAR) vs col H (Total_Loss_Amt)
#
# OUTPUT per map folder:
#   MapX_ALL.shp          - all polygons, Category field populated
#   MapX_Hit.shp          - Red    #E53935
#   MapX_Miss.shp         - Orange #FB8C00
#   MapX_False_Alarm.shp  - Yellow #FDD835
#   MapX_True_Negative.shp- Green  #43A047
#   MapX.csv              - attribute table backup
# ============================================================

import pandas as pd
import numpy as np
import geopandas as gpd
import os
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# !! UPDATE THESE PATHS !!
# ============================================================

# Your Bangladesh ADM3 polygon shapefile
SHP_INPUT = r"C:\Users\Ibrahim\Desktop\bd shape file\Shape files bd\bgd_adm_bbs_20201113_SHP\bgd_admbnda_adm3_bbs_20201113.shp"

# The field in YOUR shapefile that contains the ADM3 pcode
# Open the attribute table in ArcGIS and check the exact column name
# Common values: 'ADM3_PCODE', 'adm3_pcode', 'Pcode', 'GID_3'
SHP_PCODE_FIELD = "ADM3_PCODE"

FILE_SAT = r"C:\Users\Ibrahim\Desktop\Github_test\Forecasted_Impact.xlsx"
FILE_DDM = r"C:\Users\Ibrahim\Desktop\Github_test\Remal_ddm.xlsx"
OUT_DIR  = r"C:\Users\Ibrahim\Desktop\Github_test\results\Hit_Map"

F1_SHEET       = "Remal"
F2_HOUSE_SHEET = "House"
F2_AGRI_SHEET  = "Agriculture"

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# THRESHOLDS
# ============================================================
HOUSE_THRESHOLD = 0.0   # P5 of Norm_Impact_House
AGRI_THRESHOLD  = 0.0   # P5 of Norm_Impact_fAPAR

# ============================================================
# CATEGORY COLOURS
# ============================================================
CAT_COLORS = {
    "Hit"           : "#E53935",
    "Miss"          : "#FB8C00",
    "False Alarm"   : "#FDD835",
    "True Negative" : "#43A047",
}
CAT_ORDER = ["Hit", "Miss", "False Alarm", "True Negative"]

# ============================================================
# HELPER
# ============================================================
def clean_admin(x):
    if pd.isna(x): return np.nan
    return " ".join(str(x).strip().split())

# ============================================================
# LOAD SHAPEFILE
# ============================================================
print("Loading shapefile ...")
gdf_base = gpd.read_file(SHP_INPUT)
print(f"  Rows      : {len(gdf_base)}")
print(f"  CRS       : {gdf_base.crs}")
print(f"  Columns   : {list(gdf_base.columns)}")

if SHP_PCODE_FIELD not in gdf_base.columns:
    raise ValueError(
        f"\nField '{SHP_PCODE_FIELD}' not found in shapefile.\n"
        f"Available columns: {list(gdf_base.columns)}\n"
        f"Set SHP_PCODE_FIELD to the correct column name above."
    )

# Normalise pcode to uppercase string for safe joining
gdf_base["_pcode"] = gdf_base[SHP_PCODE_FIELD].astype(str).str.strip().str.upper()
print(f"  Pcode sample : {gdf_base['_pcode'].head(5).tolist()}")

# ============================================================
# LOAD & CLEAN SATELLITE DATA
# ============================================================
print("\nLoading satellite data ...")
df_sat = pd.read_excel(FILE_SAT, sheet_name=F1_SHEET)
df_sat.columns = [str(c).strip() for c in df_sat.columns]
df_sat = (df_sat
          .dropna(subset=["District", "Upazila"])
          .drop_duplicates(subset=["District", "Upazila"], keep="first")
          .reset_index(drop=True))
for col in ["District", "Upazila"]:
    df_sat[col] = df_sat[col].apply(clean_admin)

# Normalise ADM3_PCODE to match shapefile
df_sat["_pcode"] = df_sat["ADM3_PCODE"].astype(str).str.strip().str.upper()
print(f"  Rows         : {len(df_sat)}")
print(f"  Pcode sample : {df_sat['_pcode'].head(5).tolist()}")

# Verify pcode overlap with shapefile
overlap = set(df_sat["_pcode"]) & set(gdf_base["_pcode"])
print(f"  Matching pcodes between sat file and shapefile: "
      f"{len(overlap)} / {len(df_sat)}")
if len(overlap) == 0:
    raise ValueError(
        "No ADM3_PCODE values match between the satellite file and shapefile.\n"
        f"Sat sample  : {df_sat['_pcode'].head(5).tolist()}\n"
        f"Shape sample: {gdf_base['_pcode'].head(5).tolist()}\n"
        "Check SHP_PCODE_FIELD is correct."
    )

# ============================================================
# LOAD & CLEAN DDM DATA
# ============================================================
print("\nLoading DDM House data ...")
raw_h = pd.read_excel(FILE_DDM, sheet_name=F2_HOUSE_SHEET, header=None)
f2h   = raw_h.iloc[2:, [0, 1, 5, 9]].copy().reset_index(drop=True)
f2h.columns = ["District", "Upazila", "No_Total", "Amt_Total"]
f2h["District"] = f2h["District"].ffill()
f2h = f2h.dropna(subset=["District", "Upazila"])
for c in ["No_Total", "Amt_Total"]:
    f2h[c] = pd.to_numeric(f2h[c], errors="coerce").fillna(0)
for col in ["District", "Upazila"]:
    f2h[col] = f2h[col].apply(clean_admin)

print("\nLoading DDM Agriculture data ...")
raw_a = pd.read_excel(FILE_DDM, sheet_name=F2_AGRI_SHEET, header=None)
f2a   = raw_a.iloc[2:, [0, 1, 6, 7]].copy().reset_index(drop=True)
f2a.columns = ["District", "Upazila", "Total_Loss_Land", "Total_Loss_Amt"]
f2a["District"] = f2a["District"].ffill()
f2a = f2a.dropna(subset=["District", "Upazila"])
for c in ["Total_Loss_Land", "Total_Loss_Amt"]:
    f2a[c] = pd.to_numeric(f2a[c], errors="coerce").fillna(0)
for col in ["District", "Upazila"]:
    f2a[col] = f2a[col].apply(clean_admin)

# ============================================================
# MERGE SAT + DDM
# District+Upazila together is unique in DDM (verified).
# After this merge, ADM3_PCODE from sat is correctly attached
# to each District+Upazila pair — no name ambiguity possible
# because the sat file already has unique pcodes per row.
# ============================================================
m_house = pd.merge(df_sat, f2h, on=["District", "Upazila"], how="inner")
m_agri  = pd.merge(df_sat, f2a, on=["District", "Upazila"], how="inner")
print(f"\nMerged -> House: {len(m_house)} rows   Agri: {len(m_agri)} rows")

# ============================================================
# CATEGORY ASSIGNMENT
# ============================================================
def assign_category(imp, dmg, threshold):
    fd   = imp > threshold
    od   = dmg > 0
    cats = pd.Series("", index=imp.index, dtype=str)
    cats[ fd &  od] = "Hit"
    cats[~fd &  od] = "Miss"
    cats[ fd & ~od] = "False Alarm"
    cats[~fd & ~od] = "True Negative"
    return cats

# ============================================================
# BUILD ATTRIBUTE TABLE
# ============================================================
def build_attr(df, impact_col, damage_col, threshold,
               sat_col_ref, ddm_col_ref, map_label):
    imp  = pd.to_numeric(df[impact_col], errors="coerce")
    dmg  = pd.to_numeric(df[damage_col], errors="coerce").fillna(0)
    cats = assign_category(imp, dmg, threshold)

    # _pcode is the normalised join key — matches shapefile
    attr = pd.DataFrame({
        "_pcode"     : df["_pcode"].values,       # join key
        "ADM3_PCODE" : df["ADM3_PCODE"].values,   # original for reference
        "District"   : df["District"].values,
        "Upazila"    : df["Upazila"].values,
        "Sat_Impact" : imp.round(6).values,
        "DDM_Value"  : dmg.values,
        "Threshold"  : threshold,
        "Sat_Col"    : sat_col_ref,
        "DDM_Col"    : ddm_col_ref,
        "Forecast"   : (imp > threshold).map(
                          {True: "Damage", False: "No Impact"}).values,
        "Observed"   : (dmg > 0).map(
                          {True: "Damage", False: "No Impact"}).values,
        "Category"   : cats.values,
        "Color_Hex"  : cats.map(CAT_COLORS).values,
    })

    print(f"\n  {map_label}")
    print(f"  Threshold = {threshold}  "
          f"(Sat col {sat_col_ref} vs DDM col {ddm_col_ref})")
    for cat in CAT_ORDER:
        n = (cats == cat).sum()
        print(f"    {cat:15s}: {n:3d}  {CAT_COLORS[cat]}")
    print(f"    Total matched : {len(attr)}")
    return attr

# ============================================================
# MERGE ATTRIBUTES + SHAPEFILE, EXPORT
# JOIN KEY: _pcode (ADM3_PCODE normalised)
# This guarantees Satkhira-Kaliganj (BD555239) never gets
# mixed with Lalmonirhat-Kaliganj (BD408747)
# ============================================================
def export_map(attr_df, gdf_base, map_name, map_dir):
    os.makedirs(map_dir, exist_ok=True)

    # Left join on _pcode: every polygon kept, study rows get category
    gdf = gdf_base.merge(
        attr_df,
        on       = "_pcode",
        how      = "left",
        suffixes = ("", "_sat")
    )

    matched   = gdf["Category"].notna() & (gdf["Category"] != "")
    print(f"  Polygons total    : {len(gdf)}")
    print(f"  Matched to study  : {matched.sum()}")
    print(f"  Not in study area : {(~matched).sum()}")

    # Fill blanks for non-study polygons
    gdf["Category"]  = gdf["Category"].fillna("Not in Study")
    gdf["Color_Hex"] = gdf["Color_Hex"].fillna("#D9D9D9")

    # ── 1. Combined: all polygons
    all_path = os.path.join(map_dir, f"{map_name}_ALL.shp")
    gdf.to_file(all_path, encoding="utf-8")
    print(f"  Saved ALL         : {all_path}")

    # ── 2. Separate layer per category (study upazilas only)
    for cat in CAT_ORDER:
        sub      = gdf[gdf["Category"] == cat].copy()
        safename = cat.replace(" ", "_")
        outpath  = os.path.join(map_dir, f"{map_name}_{safename}.shp")
        if len(sub) > 0:
            sub.to_file(outpath, encoding="utf-8")
            print(f"  Saved {cat:15s}: {outpath}  (n={len(sub)})")
        else:
            print(f"  Skipped {cat:15s}: 0 rows")

    # ── 3. CSV backup (no geometry)
    csv_path = os.path.join(map_dir, f"{map_name}.csv")
    attr_df.drop(columns=["_pcode"], errors="ignore").to_csv(
        csv_path, index=False, encoding="utf-8")
    print(f"  Saved CSV         : {csv_path}")

# ============================================================
# RUN ALL 4 MAPS
# ============================================================
print("\n" + "="*60)
print("BUILDING POLYGON SHAPEFILES FOR 4 MAPS")
print("="*60)

MAPS = [
    dict(df=m_house, impact_col="Norm_Impact_House", damage_col="No_Total",
         threshold=HOUSE_THRESHOLD, sat_col_ref="AF", ddm_col_ref="F",
         map_name="Map1_House_NoTotal",
         map_label="House: Norm_Impact_House vs No. Damaged Houses (col AF vs F)"),
    dict(df=m_house, impact_col="Norm_Impact_House", damage_col="Amt_Total",
         threshold=HOUSE_THRESHOLD, sat_col_ref="AF", ddm_col_ref="J",
         map_name="Map2_House_AmtTotal",
         map_label="House: Norm_Impact_House vs Repair Amount (col AF vs J)"),
    dict(df=m_agri,  impact_col="Norm_Impact_fAPAR", damage_col="Total_Loss_Land",
         threshold=AGRI_THRESHOLD,  sat_col_ref="W",  ddm_col_ref="G",
         map_name="Map3_Agri_LossLand",
         map_label="Agri: Norm_Impact_fAPAR vs Total Land Loss (col W vs G)"),
    dict(df=m_agri,  impact_col="Norm_Impact_fAPAR", damage_col="Total_Loss_Amt",
         threshold=AGRI_THRESHOLD,  sat_col_ref="W",  ddm_col_ref="H",
         map_name="Map4_Agri_LossAmt",
         map_label="Agri: Norm_Impact_fAPAR vs Total Loss Amount (col W vs H)"),
]

all_attrs = []
for m in MAPS:
    print(f"\n{'='*60}")
    print(f"Processing {m['map_name']} ...")
    attr = build_attr(
        df          = m["df"],
        impact_col  = m["impact_col"],
        damage_col  = m["damage_col"],
        threshold   = m["threshold"],
        sat_col_ref = m["sat_col_ref"],
        ddm_col_ref = m["ddm_col_ref"],
        map_label   = m["map_label"],
    )
    attr["Map"] = m["map_name"]
    all_attrs.append(attr)

    export_map(
        attr_df  = attr,
        gdf_base = gdf_base,
        map_name = m["map_name"],
        map_dir  = os.path.join(OUT_DIR, m["map_name"]),
    )

# Master CSV
master_path = os.path.join(OUT_DIR, "Remal_All_Maps_Master.csv")
(pd.concat(all_attrs, ignore_index=True)
   .drop(columns=["_pcode"], errors="ignore")
   .to_csv(master_path, index=False, encoding="utf-8"))
print(f"\nMaster CSV: {master_path}")

print("""
============================================================
HOW TO USE IN ARCGIS PRO
============================================================
Option A - Separate layers (recommended, toggle each on/off):
  1. Add each *_Hit.shp, *_Miss.shp, *_False_Alarm.shp,
     *_True_Negative.shp as separate layers
  2. Set fill colour per layer:
       Hit           #E53935  (Red)
       Miss          #FB8C00  (Orange)
       False Alarm   #FDD835  (Yellow)
       True Negative #43A047  (Green)
  3. Toggle layers in Contents panel independently

Option B - Single layer, style by Category field:
  1. Add *_ALL.shp
  2. Symbology -> Unique Values -> Field: Category
  3. Assign colours above per class
  4. 'Not in Study' polygons -> set to grey or no fill

IMPORTANT: Join is on ADM3_PCODE - district-upazila name
duplicates (Kaliganj, Kotwali, etc.) are handled correctly.
============================================================
""")

Loading shapefile ...
  Rows      : 544
  CRS       : EPSG:4326
  Columns   : ['Shape_Leng', 'Shape_Area', 'ADM3_EN', 'ADM3_PCODE', 'ADM3_REF', 'ADM3ALT1EN', 'ADM3ALT2EN', 'ADM2_EN', 'ADM2_PCODE', 'ADM1_EN', 'ADM1_PCODE', 'ADM0_EN', 'ADM0_PCODE', 'date', 'validOn', 'validTo', 'geometry']
  Pcode sample : ['BD404104', 'BD302602', 'BD501006', 'BD555202', 'BD100602']

Loading satellite data ...
  Rows         : 544
  Pcode sample : ['BD100409', 'BD100419', 'BD100428', 'BD100447', 'BD100485']
  Matching pcodes between sat file and shapefile: 544 / 544

Loading DDM House data ...

Loading DDM Agriculture data ...

Merged -> House: 125 rows   Agri: 125 rows

BUILDING POLYGON SHAPEFILES FOR 4 MAPS

Processing Map1_House_NoTotal ...

  House: Norm_Impact_House vs No. Damaged Houses (col AF vs F)
  Threshold = 0.0  (Sat col AF vs DDM col F)
    Hit            : 105  #E53935
    Miss           :   0  #FB8C00
    False Alarm    :  19  #FDD835
    True Negative  :   1  #43A047
    Total matched : 

In [4]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────
# FILE PATHS  — update for each cyclone
# ──────────────────────────────────────────────────────────────
FILE_SAT = r"C:\Users\Ibrahim\Desktop\Github_test\Forecasted_Impact.xlsx"
FILE_DDM = r"C:\Users\Ibrahim\Desktop\Github_test\Remal_ddm.xlsx"
OUT_EXCEL = r"C:\Users\Ibrahim\Desktop\Github_test\results\DDM_Catagorization\Remal_DDM_KappaCalibrated.xlsx"
CYCLONE   = "Remal"

F1_SHEET       = "Remal"
F2_HOUSE_SHEET = "House"
F2_AGRI_SHEET  = "Agriculture"

os.makedirs(os.path.dirname(OUT_EXCEL), exist_ok=True)

# ──────────────────────────────────────────────────────────────
# SATELLITE FORECAST THRESHOLDS  (fixed, expert-defined — do not
# tune these against DDM data). Each impact column has its OWN
# set of boundaries — FILL IN THE CORRECT VALUES BELOW.
# ──────────────────────────────────────────────────────────────
SAT_THRESHOLDS = {
    "Norm_Impact_House": {
        "low":      0.07747,    # <-- INPUT: No Impact / Low boundary for House
        "moderate": 0.2994,    # <-- INPUT: Low / Moderate boundary for House
        "high":     0.6503,   # <-- INPUT: Moderate / High boundary for House
    },
    "Norm_Impact_fAPAR": {
        "low":      0.2321,    # <-- INPUT: No Impact / Low boundary for Agriculture/fAPAR
        "moderate": 0.29968,    # <-- INPUT: Low / Moderate boundary for Agriculture/fAPAR
        "high":     0.6657,    # <-- INPUT: Moderate / High boundary for Agriculture/fAPAR
    },
}

def sat_class(v, impact_col):
    """Classify a satellite impact value using the threshold set
    that belongs to its own impact_col (House and Agriculture use
    different, independently expert-defined boundaries)."""
    v = float(v)
    t = SAT_THRESHOLDS[impact_col]
    if v < t["low"]:
        return "No Impact"
    if v < t["moderate"]:
        return "Low"
    if v < t["high"]:
        return "Moderate"
    return "High"

SAT_IDX = {"No Impact": 0, "Low": 1, "Moderate": 2, "High": 3}
DDM_IDX = {"No Impact": 0, "Low": 1, "Medium": 2, "High": 3}

# ──────────────────────────────────────────────────────────────
# EXCEL STYLES
# ──────────────────────────────────────────────────────────────
NAVY="1F3864"; BLUE="2E75B6"; LBLUE="D6E4F0"; LGREY="F2F2F2"; WHITE="FFFFFF"
GREEN="C8E6C9"; AMBER="FFE0B2"; RED_BG="FFCDD2"; YELLOW="FFF9C4"
CAT_FILL={"Hit":"A5D6A7","Miss":"EF9A9A","False Alarm":"FFE082","True Negative":"B3E5FC"}
MATCH_FILL={"Exact":GREEN,"Adjacent":YELLOW,"Off":RED_BG,"":WHITE}
SEV_FILL={"No Impact":"F5F5F5","Low":"E3F2FD","Medium":YELLOW,"High":RED_BG}

thin=Side(style="thin",color="CCCCCC")
TB=Border(left=thin,right=thin,top=thin,bottom=thin)
C=Alignment(horizontal="center",vertical="center",wrap_text=True)
L=Alignment(horizontal="left",  vertical="center",wrap_text=True)

def hf(): return Font(bold=True,size=9,name="Arial",color="FFFFFF")
def bf(bold=False,color="000000"): return Font(bold=bold,size=9,name="Arial",color=color)
def tf(): return Font(bold=True,size=13,name="Arial",color="FFFFFF")
def pf(h): return PatternFill("solid",fgColor=h)

def clean_admin(x):
    if pd.isna(x): return np.nan
    return " ".join(str(x).strip().split())

# ──────────────────────────────────────────────────────────────
# CORE: weighted kappa
# ──────────────────────────────────────────────────────────────
def weighted_kappa(a, b, K=4):
    O = np.zeros((K, K))
    for x, y in zip(a, b):
        O[int(x)][int(y)] += 1
    N = len(a)
    if N == 0: return 0.0
    ra = O.sum(1); ca = O.sum(0)
    w = np.array([[(i-j)**2/(K-1)**2 for j in range(K)] for i in range(K)])
    num = (w * O).sum()
    den = (w * np.outer(ra, ca) / N).sum()
    return float(1 - num/den) if den else 0.0

# ──────────────────────────────────────────────────────────────
# CORE: exhaustive kappa-calibration search (DDM side only —
# the satellite classes above are held fixed throughout)
# ──────────────────────────────────────────────────────────────
def find_optimal_cuts(sat_idx_arr, dmg_arr):
    """
    Exhaustively tests every valid pair (low_cut, high_cut) drawn from
    the actual unique non-zero damage values in dmg_arr.
    Returns the pair that maximises quadratic-weighted kappa against
    the fixed sat_idx_arr (satellite classes never move here).
    Also returns the full results table for the sensitivity curve sheet.
    """
    nz_vals = sorted(set(dmg_arr[dmg_arr > 0].tolist()))
    n_vals  = len(nz_vals)

    if n_vals < 2:
        return np.nan, np.nan, np.nan, pd.DataFrame()

    best_k    = -np.inf
    best_lc   = np.nan
    best_hc   = np.nan
    rows      = []

    total_combos = n_vals * (n_vals - 1) // 2
    printed = set()
    milestones = {int(total_combos * f) for f in [0.25, 0.5, 0.75]}

    combo = 0
    for i, lc in enumerate(nz_vals[:-1]):
        for hc in nz_vals[i+1:]:
            ddm_idx = np.array([
                0 if v <= 0 else (1 if v <= lc else (2 if v <= hc else 3))
                for v in dmg_arr
            ])
            k = weighted_kappa(sat_idx_arr, ddm_idx)
            rows.append({"Low_Cut": lc, "High_Cut": hc, "Kappa": k})
            if k > best_k:
                best_k, best_lc, best_hc = k, lc, hc
            combo += 1
            if combo in milestones and combo not in printed:
                printed.add(combo)
                print(f"      {combo}/{total_combos} combinations tested ...")

    df_all = pd.DataFrame(rows).sort_values("Kappa", ascending=False).reset_index(drop=True)
    return float(best_lc), float(best_hc), float(best_k), df_all

# ──────────────────────────────────────────────────────────────
# HELPERS: DDM classification and verification
# ──────────────────────────────────────────────────────────────
def ddm_class(v, low_cut, high_cut):
    if v <= 0:        return "No Impact"
    if v <= low_cut:  return "Low"
    if v <= high_cut: return "Medium"
    return "High"

def tertile_cuts(series):
    nz = series[series > 0].values
    if len(nz) == 0: return 0.0, 0.0
    return float(np.percentile(nz, 33.33)), float(np.percentile(nz, 66.67))

def binary_category(sat_cls, ddm_cls):
    fd = sat_cls != "No Impact"
    od = ddm_cls != "No Impact"
    if fd and od:     return "Hit"
    if not fd and od: return "Miss"
    if fd and not od: return "False Alarm"
    return "True Negative"

def severity_match(sat_cls, ddm_cls, binary_cat):
    if binary_cat != "Hit": return ""
    d = abs(SAT_IDX.get(sat_cls, 0) - DDM_IDX.get(ddm_cls, 0))
    return "Exact" if d == 0 else ("Adjacent" if d == 1 else "Off")

def build_result_df(df_merged, impact_col, damage_col, low_cut, high_cut, unit):
    df = df_merged.copy()
    imp = pd.to_numeric(df[impact_col], errors="coerce")
    dmg = pd.to_numeric(df[damage_col], errors="coerce").fillna(0)
    df["Sat_Impact"]         = imp.round(6)
    # NOTE: pass impact_col through so each row is classified using
    # the threshold set that belongs to THIS variable (House vs Agri)
    df["Forecast_Class"]     = imp.apply(lambda v: sat_class(v, impact_col))
    df["DDM_Value"]          = dmg
    df["DDM_Unit"]           = unit
    df["DDM_Severity_Class"] = dmg.apply(lambda v: ddm_class(v, low_cut, high_cut))
    df["Binary_Category"]    = [binary_category(fc, dc)
                                 for fc, dc in zip(df["Forecast_Class"], df["DDM_Severity_Class"])]
    df["Severity_Match"]     = [severity_match(fc, dc, bc)
                                 for fc, dc, bc in zip(df["Forecast_Class"],
                                                       df["DDM_Severity_Class"],
                                                       df["Binary_Category"])]
    df["Low_Cut"]  = low_cut
    df["High_Cut"] = high_cut
    out = df[["ADM3_PCODE","District","Upazila","Sat_Impact","Forecast_Class",
              "DDM_Value","DDM_Unit","DDM_Severity_Class","Binary_Category",
              "Severity_Match","Low_Cut","High_Cut"]].copy()
    return out

def compute_stats(out_df):
    si = [SAT_IDX.get(c, 0) for c in out_df["Forecast_Class"]]
    di = [DDM_IDX.get(c, 0) for c in out_df["DDM_Severity_Class"]]
    kappa = weighted_kappa(si, di)
    exact = np.mean([a == b for a, b in zip(si, di)])
    adj   = np.mean([abs(a-b) <= 1 for a, b in zip(si, di)])
    cats  = out_df["Binary_Category"]
    hit, miss = (cats=="Hit").sum(), (cats=="Miss").sum()
    fa,  tn   = (cats=="False Alarm").sum(), (cats=="True Negative").sum()
    hits_df = out_df[out_df["Binary_Category"]=="Hit"]
    return dict(kappa=kappa, exact=exact, adj=adj,
                hit=hit, miss=miss, fa=fa, tn=tn,
                hits_exact=(hits_df["Severity_Match"]=="Exact").sum(),
                hits_adj=(hits_df["Severity_Match"]=="Adjacent").sum(),
                hits_off=(hits_df["Severity_Match"]=="Off").sum(),
                sat_idx=si, ddm_idx=di)

# ──────────────────────────────────────────────────────────────
# LOAD DATA
# ──────────────────────────────────────────────────────────────
print(f"Loading data for Cyclone {CYCLONE} ...")

df_sat = pd.read_excel(FILE_SAT, sheet_name=F1_SHEET)
df_sat.columns = [str(c).strip() for c in df_sat.columns]
df_sat = (df_sat.dropna(subset=["District","Upazila"])
                .drop_duplicates(subset=["District","Upazila"], keep="first")
                .reset_index(drop=True))
for col in ["District","Upazila"]:
    df_sat[col] = df_sat[col].apply(clean_admin)

raw_h = pd.read_excel(FILE_DDM, sheet_name=F2_HOUSE_SHEET, header=None)
f2h = raw_h.iloc[2:, [0,1,5,9]].copy().reset_index(drop=True)
f2h.columns = ["District","Upazila","No_Total","Amt_Total"]
f2h["District"] = f2h["District"].ffill()
f2h = f2h.dropna(subset=["District","Upazila"])
for c in ["No_Total","Amt_Total"]:
    f2h[c] = pd.to_numeric(f2h[c], errors="coerce").fillna(0)
for col in ["District","Upazila"]:
    f2h[col] = f2h[col].apply(clean_admin)

raw_a = pd.read_excel(FILE_DDM, sheet_name=F2_AGRI_SHEET, header=None)
f2a = raw_a.iloc[2:, [0,1,6,7]].copy().reset_index(drop=True)
f2a.columns = ["District","Upazila","Total_Loss_Land","Total_Loss_Amt"]
f2a["District"] = f2a["District"].ffill()
f2a = f2a.dropna(subset=["District","Upazila"])
for c in ["Total_Loss_Land","Total_Loss_Amt"]:
    f2a[c] = pd.to_numeric(f2a[c], errors="coerce").fillna(0)
for col in ["District","Upazila"]:
    f2a[col] = f2a[col].apply(clean_admin)

m_house = pd.merge(df_sat, f2h, on=["District","Upazila"], how="inner")
m_agri  = pd.merge(df_sat, f2a, on=["District","Upazila"], how="inner")
print(f"Merged -> House: {len(m_house)}   Agri: {len(m_agri)}\n")

MAPS = [
    dict(df=m_house, impact_col="Norm_Impact_House", damage_col="No_Total",
         unit="Damaged Households", sat_col="AF", ddm_col="F",
         sheet="House_NoTotal",
         label="House: No. Damaged Households (Sat col AF vs DDM col F)"),
    dict(df=m_house, impact_col="Norm_Impact_House", damage_col="Amt_Total",
         unit="BDT (Repair Amount)", sat_col="AF", ddm_col="J",
         sheet="House_AmtTotal",
         label="House: Repair Amount BDT (Sat col AF vs DDM col J)"),
    dict(df=m_agri,  impact_col="Norm_Impact_fAPAR", damage_col="Total_Loss_Land",
         unit="Hectares (Land Loss)", sat_col="W", ddm_col="G",
         sheet="Agri_LossLand",
         label="Agriculture: Land Loss ha (Sat col W vs DDM col G)"),
    dict(df=m_agri,  impact_col="Norm_Impact_fAPAR", damage_col="Total_Loss_Amt",
         unit="BDT (Loss Amount)", sat_col="W", ddm_col="H",
         sheet="Agri_LossAmt",
         label="Agriculture: Loss Amount BDT (Sat col W vs DDM col H)"),
]

# Guard: refuse to run with unfilled fAPAR thresholds
for col, t in SAT_THRESHOLDS.items():
    if any(v is None for v in t.values()):
        raise ValueError(
            f"SAT_THRESHOLDS['{col}'] still has a None value. "
            f"Fill in the expert-defined low/moderate/high boundaries "
            f"for this variable before running the script."
        )

# ──────────────────────────────────────────────────────────────
# RUN CALIBRATION FOR EACH VARIABLE
# ──────────────────────────────────────────────────────────────
all_results = {}

for m in MAPS:
    print(f"{'='*60}")
    print(f"Calibrating: {m['label']}")
    print(f"  Using satellite thresholds for {m['impact_col']}: {SAT_THRESHOLDS[m['impact_col']]}")

    imp = pd.to_numeric(m["df"][m["impact_col"]], errors="coerce")
    dmg = pd.to_numeric(m["df"][m["damage_col"]], errors="coerce").fillna(0)
    # NOTE: uses this variable's OWN threshold set, not a shared/global one
    sat_idx_arr = imp.apply(lambda v: sat_class(v, m["impact_col"])).map(SAT_IDX).values
    dmg_arr     = dmg.values

    # ── Tertile baseline (independent, not tuned to satellite) ─
    t_lc, t_hc = tertile_cuts(dmg)
    ddm_t = np.array([0 if v<=0 else (1 if v<=t_lc else (2 if v<=t_hc else 3))
                       for v in dmg_arr])
    k_tertile = weighted_kappa(sat_idx_arr, ddm_t)

    # ── Exhaustive DDM-side calibration ────────────────────────
    print(f"  Running exhaustive search ...")
    opt_lc, opt_hc, k_optimal, df_sweep = find_optimal_cuts(sat_idx_arr, dmg_arr)

    print(f"  Tertile    (P33/P67): lc={t_lc:>12,.1f}  hc={t_hc:>14,.1f}  kappa={k_tertile:+.4f}")
    print(f"  Calibrated (exhaustive): lc={opt_lc:>12,.1f}  hc={opt_hc:>14,.1f}  kappa={k_optimal:+.4f}")
    print(f"  Kappa improvement     : {k_optimal - k_tertile:+.4f}")

    # ── Build classified output ───────────────────────────────
    out_opt   = build_result_df(m["df"], m["impact_col"], m["damage_col"],
                                 opt_lc, opt_hc, m["unit"])
    out_tert  = build_result_df(m["df"], m["impact_col"], m["damage_col"],
                                 t_lc,   t_hc,   m["unit"])

    stats_opt  = compute_stats(out_opt)
    stats_tert = compute_stats(out_tert)

    # Percentile positions of the calibrated cuts
    nz = dmg_arr[dmg_arr > 0]
    pct_lc = float(np.mean(nz <= opt_lc) * 100)
    pct_hc = float(np.mean(nz <= opt_hc) * 100)

    all_results[m["sheet"]] = dict(
        out_opt=out_opt, out_tert=out_tert,
        stats_opt=stats_opt, stats_tert=stats_tert,
        opt_lc=opt_lc, opt_hc=opt_hc, k_optimal=k_optimal,
        t_lc=t_lc, t_hc=t_hc, k_tertile=k_tertile,
        pct_lc=pct_lc, pct_hc=pct_hc,
        df_sweep=df_sweep, m=m
    )
    print()

# ──────────────────────────────────────────────────────────────
# BUILD EXCEL
# ──────────────────────────────────────────────────────────────
print("Writing Excel ...")
wb = Workbook()
wb.remove(wb.active)

# ════════════════════════════════════════════════════════════
# SHEET 1 — Summary
# ════════════════════════════════════════════════════════════
ws_sum = wb.create_sheet("Summary")
ws_sum.merge_cells("A1:T1")
ws_sum["A1"] = (f"Cyclone {CYCLONE}  —  Kappa-Calibrated DDM Classification  "
                f"vs Tertile Baseline")
ws_sum["A1"].font = tf()
ws_sum["A1"].fill = pf(NAVY)
ws_sum["A1"].alignment = C
ws_sum.row_dimensions[1].height = 26

ws_sum.merge_cells("A2:T2")
ws_sum["A2"] = ("Calibrated: exhaustive search over all unique damage value pairs to maximise "
                "quadratic-weighted kappa between the FIXED, expert-defined satellite forecast "
                "class (own threshold set per variable — House and Agriculture/fAPAR use "
                "different boundaries) and the DDM severity class. "
                "Tertile: standard P33/P67 split for comparison. "
                "Green kappa = Substantial (>=0.60) | Amber = Moderate (>=0.40) | Red = Fair (<0.40).")
ws_sum["A2"].font = Font(italic=True,size=8,name="Arial",color="444444")
ws_sum["A2"].fill = pf(LBLUE)
ws_sum["A2"].alignment = L
ws_sum.row_dimensions[2].height = 30

HDRS = ["Variable","Sat Col","DDM Col",
        "Sat Thresholds Used",
        "Calibrated Low Cut","Calibrated High Cut","Low Pct","High Pct","Calibrated Kappa",
        "Exact %(Cal)","Within-1%(Cal)",
        "Tertile Low Cut","Tertile High Cut","Tertile Kappa",
        "Exact%(Tert)","Within-1%(Tert)",
        "Kappa Gain","Improvement?"]
for ci,h in enumerate(HDRS,1):
    c=ws_sum.cell(3,ci); c.value=h; c.font=hf(); c.fill=pf(NAVY); c.alignment=C; c.border=TB
ws_sum.row_dimensions[3].height=30

def kappa_fill(k):
    return GREEN if k>=0.60 else (AMBER if k>=0.40 else RED_BG)

r=4
for sn,res in all_results.items():
    m=res["m"]; so=res["stats_opt"]; st=res["stats_tert"]
    gain = res["k_optimal"] - res["k_tertile"]
    t = SAT_THRESHOLDS[m["impact_col"]]
    thresh_str = f"{t['low']}/{t['moderate']}/{t['high']}"
    row=[
        m["label"].split("(")[0].strip(), m["sat_col"], m["ddm_col"],
        thresh_str,
        f"{res['opt_lc']:,.1f}", f"{res['opt_hc']:,.1f}",
        f"P{res['pct_lc']:.0f}", f"P{res['pct_hc']:.0f}",
        f"{res['k_optimal']:+.4f}",
        f"{so['exact']*100:.0f}%", f"{so['adj']*100:.0f}%",
        f"{res['t_lc']:,.1f}", f"{res['t_hc']:,.1f}",
        f"{res['k_tertile']:+.4f}",
        f"{st['exact']*100:.0f}%", f"{st['adj']*100:.0f}%",
        f"{gain:+.4f}",
        "Yes — use calibrated" if gain > 0.01 else "Marginal",
    ]
    for ci,v in enumerate(row,1):
        c=ws_sum.cell(r,ci); c.value=v
        c.font=bf(); c.alignment=C if ci>1 else L; c.border=TB
        if ci==9:  c.fill=pf(kappa_fill(res["k_optimal"]))
        if ci==14: c.fill=pf(kappa_fill(res["k_tertile"]))
        if ci==17:
            g=float(v.replace("+",""))
            c.fill=pf(GREEN if g>0.01 else (AMBER if g>0 else RED_BG))
    ws_sum.row_dimensions[r].height=16; r+=1

ws_sum.column_dimensions["A"].width=42
for ci,w in enumerate([7,7,16,16,16,10,10,13,11,13,16,16,13,11,13,12,16],2):
    ws_sum.column_dimensions[get_column_letter(ci+1)].width=w
ws_sum.freeze_panes="A4"

# ════════════════════════════════════════════════════════════
# DETAIL SHEETS  — calibrated classification
# ════════════════════════════════════════════════════════════
COL_HDRS=["ADM3_PCODE","District","Upazila","Sat_Impact","Forecast_Class",
          "DDM_Value","DDM_Unit","DDM_Severity_Class","Binary_Category",
          "Severity_Match","Low_Cut","High_Cut"]
COL_W=[14,18,22,12,14,16,22,18,16,16,12,12]

for idx,(sn,res) in enumerate(all_results.items(),2):
    ws=wb.create_sheet(f"{idx}. {sn}_Calibrated")
    out_df=res["out_opt"]; m=res["m"]; so=res["stats_opt"]
    t = SAT_THRESHOLDS[m["impact_col"]]

    ws.merge_cells(f"A1:{get_column_letter(len(COL_HDRS))}1")
    ws["A1"]=f"Cyclone {CYCLONE}  —  {m['label']}  [KAPPA-CALIBRATED DDM CUTS]"
    ws["A1"].font=Font(bold=True,size=11,name="Arial",color="FFFFFF")
    ws["A1"].fill=pf(NAVY); ws["A1"].alignment=L
    ws.row_dimensions[1].height=22

    ws.merge_cells(f"A2:{get_column_letter(len(COL_HDRS))}2")
    ws["A2"]=(f"Sat thresholds used ({m['impact_col']}): "
              f"No Impact<{t['low']} | Low<{t['moderate']} | Moderate<{t['high']} | High>= | "
              f"DDM calibrated cuts: Low<={res['opt_lc']:,.1f} | Medium<={res['opt_hc']:,.1f} | "
              f"High>{res['opt_hc']:,.1f} ({m['unit']})  |  "
              f"These are P{res['pct_lc']:.0f}/P{res['pct_hc']:.0f} of non-zero values  |  "
              f"kappa={res['k_optimal']:+.4f}  vs  tertile kappa={res['k_tertile']:+.4f}  "
              f"(gain={res['k_optimal']-res['k_tertile']:+.4f})  |  "
              f"Hit={so['hit']} Miss={so['miss']} FA={so['fa']} TN={so['tn']}  |  "
              f"Exact={so['exact']*100:.0f}% Within-1={so['adj']*100:.0f}%")
    ws["A2"].font=Font(italic=True,size=8,name="Arial",color="333333")
    ws["A2"].fill=pf(LBLUE); ws["A2"].alignment=L
    ws.row_dimensions[2].height=28

    for ci,h in enumerate(COL_HDRS,1):
        c=ws.cell(3,ci); c.value=h; c.font=hf(); c.fill=pf(NAVY); c.alignment=C; c.border=TB
    ws.row_dimensions[3].height=18

    for ri,(_,row) in enumerate(out_df.iterrows(),4):
        cat=str(row.get("Binary_Category","")); match=str(row.get("Severity_Match",""))
        sev=str(row.get("DDM_Severity_Class","")); fc=str(row.get("Forecast_Class",""))
        for ci,col in enumerate(COL_HDRS,1):
            val=row.get(col,""); cell=ws.cell(ri,ci)
            cell.border=TB; cell.alignment=C if ci>3 else L; cell.font=bf()
            if col in ["Sat_Impact","Low_Cut","High_Cut"]:
                cell.value=round(float(val),4) if val!="" else ""
                cell.number_format="0.0000"
            elif col=="DDM_Value":
                cell.value=int(val) if val!="" else 0; cell.number_format="#,##0"
            else: cell.value=val
            if col=="Binary_Category": cell.fill=pf(CAT_FILL.get(cat,WHITE)); cell.font=bf(bold=True)
            elif col=="Severity_Match": cell.fill=pf(MATCH_FILL.get(match,WHITE))
            elif col=="DDM_Severity_Class": cell.fill=pf(SEV_FILL.get(sev,WHITE))
            elif col=="Forecast_Class": cell.fill=pf(SEV_FILL.get(fc,WHITE))
            else: cell.fill=pf(LGREY if ri%2==0 else WHITE)
        ws.row_dimensions[ri].height=14

    for ci,w in enumerate(COL_W,1): ws.column_dimensions[get_column_letter(ci)].width=w
    ws.freeze_panes="A4"

# ════════════════════════════════════════════════════════════
# COMPARISON SHEET  — calibrated vs tertile side by side
# ════════════════════════════════════════════════════════════
ws_cmp = wb.create_sheet("Comparison_Cal_vs_Tertile")
ws_cmp.merge_cells("A1:I1")
ws_cmp["A1"]=f"Cyclone {CYCLONE}  —  Calibrated vs Tertile: Contingency Tables"
ws_cmp["A1"].font=tf(); ws_cmp["A1"].fill=pf(NAVY); ws_cmp["A1"].alignment=C
ws_cmp.row_dimensions[1].height=24

r=3
SAT_CI={"No Impact":0,"Low":1,"Moderate":2,"High":3}
DDM_CI={"No Impact":0,"Low":1,"Medium":2,"High":3}

for sn,res in all_results.items():
    m=res["m"]
    ws_cmp.merge_cells(f"A{r}:I{r}")
    ws_cmp[f"A{r}"]=m["label"]; ws_cmp[f"A{r}"].font=Font(bold=True,size=10,name="Arial",color=NAVY)
    ws_cmp[f"A{r}"].fill=pf(LBLUE); ws_cmp[f"A{r}"].alignment=L
    ws_cmp.row_dimensions[r].height=18; r+=1

    for label,out_df,kv in [
        (f"CALIBRATED  (kappa={res['k_optimal']:+.4f})",  res["out_opt"],  res["k_optimal"]),
        (f"TERTILE     (kappa={res['k_tertile']:+.4f})",  res["out_tert"], res["k_tertile"]),
    ]:
        # Sub-header
        ws_cmp.cell(r,1).value=label
        ws_cmp.cell(r,1).font=Font(bold=True,size=9,name="Arial",
                                    color=NAVY if kv==res["k_optimal"] else "888888")
        ws_cmp.cell(r,1).fill=pf(GREEN if kv==res["k_optimal"] else LGREY)
        ws_cmp.cell(r,1).border=TB; ws_cmp.row_dimensions[r].height=16; r+=1

        # Header row
        ddm_classes=["No Impact","Low","Medium","High"]
        sat_classes=["No Impact","Low","Moderate","High"]
        ws_cmp.cell(r,1).value="Obs\\Fcst"
        ws_cmp.cell(r,1).font=hf(); ws_cmp.cell(r,1).fill=pf(NAVY)
        ws_cmp.cell(r,1).border=TB; ws_cmp.cell(r,1).alignment=C
        for ci,sc in enumerate(sat_classes,2):
            c=ws_cmp.cell(r,ci); c.value=sc; c.font=hf()
            c.fill=pf(NAVY); c.alignment=C; c.border=TB
        ws_cmp.cell(r,6).value="Total"; ws_cmp.cell(r,6).font=hf()
        ws_cmp.cell(r,6).fill=pf(NAVY); ws_cmp.cell(r,6).alignment=C; ws_cmp.cell(r,6).border=TB
        ws_cmp.row_dimensions[r].height=16; r+=1

        # Matrix
        M=np.zeros((4,4),int)
        for _,row_d in out_df.iterrows():
            fi=SAT_CI.get(str(row_d["Forecast_Class"]),0)
            di=DDM_CI.get(str(row_d["DDM_Severity_Class"]),0)
            M[di][fi]+=1

        for ri2,dc in enumerate(ddm_classes):
            ws_cmp.cell(r,1).value=dc
            ws_cmp.cell(r,1).font=bf(bold=True); ws_cmp.cell(r,1).fill=pf(LGREY)
            ws_cmp.cell(r,1).border=TB; ws_cmp.cell(r,1).alignment=L
            for ci in range(4):
                cell=ws_cmp.cell(r,ci+2); cell.value=int(M[ri2][ci])
                cell.font=bf(bold=(ri2==ci)); cell.border=TB; cell.alignment=C
                cell.fill=pf("C8E6C9") if ri2==ci else pf(WHITE)
            ws_cmp.cell(r,6).value=int(M[ri2].sum())
            ws_cmp.cell(r,6).font=bf(bold=True); ws_cmp.cell(r,6).fill=pf(LGREY)
            ws_cmp.cell(r,6).border=TB; ws_cmp.cell(r,6).alignment=C
            ws_cmp.row_dimensions[r].height=14; r+=1
        r+=1

for ci,w in enumerate([28,14,14,14,14,10],1):
    ws_cmp.column_dimensions[get_column_letter(ci)].width=w

# ════════════════════════════════════════════════════════════
# SENSITIVITY SHEET  — top 50 combinations per variable
# ════════════════════════════════════════════════════════════
ws_sw = wb.create_sheet("Sensitivity_Top50")
ws_sw.merge_cells("A1:F1")
ws_sw["A1"]=f"Cyclone {CYCLONE}  —  Top 50 Cut Combinations by Kappa (per variable)"
ws_sw["A1"].font=tf(); ws_sw["A1"].fill=pf(NAVY); ws_sw["A1"].alignment=C
ws_sw.row_dimensions[1].height=24

r=3
for sn,res in all_results.items():
    m=res["m"]
    ws_sw.merge_cells(f"A{r}:F{r}")
    ws_sw[f"A{r}"]=m["label"]
    ws_sw[f"A{r}"].font=Font(bold=True,size=10,name="Arial",color=NAVY)
    ws_sw[f"A{r}"].fill=pf(LBLUE); ws_sw[f"A{r}"].alignment=L
    ws_sw.row_dimensions[r].height=18; r+=1

    for ci,h in enumerate(["Rank","Low Cut","High Cut","Kappa","Best?"],1):
        c=ws_sw.cell(r,ci); c.value=h; c.font=hf(); c.fill=pf(NAVY); c.alignment=C; c.border=TB
    ws_sw.row_dimensions[r].height=16; r+=1

    top50 = res["df_sweep"].head(50)
    for rank,(_, row_s) in enumerate(top50.iterrows(),1):
        is_best = (rank == 1)
        for ci,v in enumerate([rank, f"{row_s['Low_Cut']:,.1f}",
                                f"{row_s['High_Cut']:,.1f}",
                                f"{row_s['Kappa']:+.4f}",
                                "<-- CALIBRATED" if is_best else ""],1):
            c=ws_sw.cell(r,ci); c.value=v; c.font=bf(bold=is_best)
            c.alignment=C; c.border=TB
            c.fill=pf(GREEN if is_best else (LGREY if rank%2==0 else WHITE))
        ws_sw.row_dimensions[r].height=14; r+=1
    r+=1

for ci,w in enumerate([8,16,16,12,14],1):
    ws_sw.column_dimensions[get_column_letter(ci)].width=w

wb.save(OUT_EXCEL)
print(f"\nSaved: {OUT_EXCEL}")
print("Sheets: Summary | Per-variable calibrated | Comparison vs Tertile | Sensitivity Top50")
print()
print("RESULTS SUMMARY")
print("="*70)
for sn,res in all_results.items():
    m=res["m"]; gain=res["k_optimal"]-res["k_tertile"]
    print(f"{m['label'].split('(')[0].strip()}")
    print(f"  Calibrated : Low<={res['opt_lc']:>12,.1f}  High>{res['opt_hc']:>12,.1f}  kappa={res['k_optimal']:+.4f}")
    print(f"  Tertile    : Low<={res['t_lc']:>12,.1f}  High>{res['t_hc']:>12,.1f}  kappa={res['k_tertile']:+.4f}")
    print(f"  Gain       : {gain:+.4f}  {'-- meaningful improvement' if gain>0.01 else '-- marginal'}")
    print()

Loading data for Cyclone Remal ...
Merged -> House: 125   Agri: 125

Calibrating: House: No. Damaged Households (Sat col AF vs DDM col F)
  Using satellite thresholds for Norm_Impact_House: {'low': 0.07747, 'moderate': 0.2994, 'high': 0.6503}
  Running exhaustive search ...
      1001/4005 combinations tested ...
      2002/4005 combinations tested ...
      3003/4005 combinations tested ...
  Tertile    (P33/P67): lc=        80.7  hc=         827.1  kappa=+0.4530
  Calibrated (exhaustive): lc=     2,775.0  hc=       4,277.0  kappa=+0.6690
  Kappa improvement     : +0.2159

Calibrating: House: Repair Amount BDT (Sat col AF vs DDM col J)
  Using satellite thresholds for Norm_Impact_House: {'low': 0.07747, 'moderate': 0.2994, 'high': 0.6503}
  Running exhaustive search ...
      1262/5050 combinations tested ...
      2525/5050 combinations tested ...
      3787/5050 combinations tested ...
  Tertile    (P33/P67): lc= 2,478,950.0  hc=  37,005,752.0  kappa=+0.4596
  Calibrated (exhaustive

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
# ----------------------------
# FILE PATHS
# ----------------------------
DDM_CAT_EXCEL = r"C:\Users\Ibrahim\Desktop\Github_test\results\DDM_Catagorization/Remal_DDM_KappaCalibrated.xlsx"
OUT_DIR       = r"C:\Users\Ibrahim\Desktop\Github_test\results\DDM_Catagorization"
CYCLONE       = "Remal"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# WHICH EXCEL SHEET FEEDS EACH SUBPLOT
# (top-left, top-right, bottom-left, bottom-right — same order as the figure)
# ----------------------------
SHEETS = [
    "2. House_NoTotal_Calibrated",
    "3. House_AmtTotal_Calibrated",
    "4. Agri_LossLand_Calibrated",
    "5. Agri_LossAmt_Calibrated",
]

# ----------------------------
# EDIT TITLES HERE — one line per subplot, same order as SHEETS above.
# Change the text freely; it has no effect on which sheet is read.
# ----------------------------
TITLES = [
    "Number of Damaged Households",
    "Monitory Damage for House",
    "Total Agricultural Land Area Damaged",
    "Monitory Damage for Agricultural Land ",
]

# Set to False if you don't want "(n=..., Exact=..., Adj=..., Off=...)"
# automatically appended after each title above.
SHOW_STATS_IN_TITLE = True

# ----------------------------
# COLOR SCHEME
# ----------------------------
COLOR_EXACT    = "#639754"   # class distance 0
COLOR_ADJACENT = "#FFD301"   # class distance 1
COLOR_OFF      = "#D61F1F"   # class distance 2 or 3

SAT_IDX = {"No Impact":0,"Low":1,"Moderate":2,"High":3}
DDM_IDX = {"No Impact":0,"Low":1,"Medium":2,"High":3}

def cell_color(dist):
    if dist == 0:
        return COLOR_EXACT
    if dist == 1:
        return COLOR_ADJACENT
    return COLOR_OFF  # dist == 2 or 3

# ----------------------------
# HELPER FUNCTIONS
# ----------------------------
def build_confusion_matrix(df):
    M = np.zeros((4,4))
    for _, r in df.iterrows():
        fi = SAT_IDX.get(str(r["Forecast_Class"]).strip(),0)
        di = DDM_IDX.get(str(r["DDM_Severity_Class"]).strip(),0)
        M[di,fi] += 1
    return M

# ----------------------------
# PLOT HEATMAP
# ----------------------------
fig, axes = plt.subplots(2,2, figsize=(14,12))
fig.patch.set_facecolor("#F8F9FA")

for ax, sheet_name, title_text in zip(axes.flatten(), SHEETS, TITLES):
    df = pd.read_excel(DDM_CAT_EXCEL, sheet_name=sheet_name, engine="openpyxl")

    # Rename columns based on your sheet structure
    df = df.rename(columns={
        df.columns[4]:"Forecast_Class",
        df.columns[7]:"DDM_Severity_Class"
    })

    M = build_confusion_matrix(df)
    n = M.sum()
    exact = sum(M[i,i] for i in range(4))
    adj   = sum(M[i,i+1] + M[i+1,i] for i in range(3))
    off   = n - exact - adj

    # Draw cells
    for di in range(4):
        for fi in range(4):
            dist = abs(di-fi)
            rect = plt.Rectangle((fi-0.45, di-0.45), 0.9, 0.9, facecolor=cell_color(dist))
            ax.add_patch(rect)
            if M[di,fi] > 0:
                ax.text(fi, di, str(int(M[di,fi])), ha="center", va="center")

    ax.set_xticks([0,1,2,3])
    ax.set_yticks([0,1,2,3])
    ax.set_xticklabels(["No Impact","Low","Moderate","High"])
    ax.set_yticklabels(["No Impact","Low","Medium","High"])
    ax.set_xlabel("Forecasted Impact ")
    ax.set_ylabel("DDM Observed")
    ax.set_xlim(-0.52,3.52)
    ax.set_ylim(3.52,-0.52)

    if SHOW_STATS_IN_TITLE:
        full_title = f"{title_text} (n={int(n)}, Exact={int(exact)}, Adj={int(adj)}, Off={int(off)})"
    else:
        full_title = title_text
    ax.set_title(full_title, fontsize=10, fontweight="bold")

# Legend
patches = [mpatches.Patch(facecolor=COLOR_EXACT,    label="Exact"),
           mpatches.Patch(facecolor=COLOR_ADJACENT, label="Adjacent"),
           mpatches.Patch(facecolor=COLOR_OFF,       label="Off")]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=10, bbox_to_anchor=(0.5,-0.02), framealpha=0.95)
plt.tight_layout(rect=[0,0.04,1,0.99])
out_path = os.path.join(OUT_DIR, f"{CYCLONE}_Hit_Performance_Heatmap.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Heatmap saved at: {out_path}")

Heatmap saved at: C:\Users\Ibrahim\Desktop\Github_test\results\DDM_Catagorization\Remal_Hit_Performance_Heatmap.png
